In [1]:
import pandas as pd
import numpy as np

# aggregated_results is produced by gencsv.py, which read all SUMMARY_* files and parse them into one CSV
# now contains 10 reps data for all 6 subjects, with ENHANCED/REDUCED results for FIX scenario, and deepseek-r1 results
data_path = "aggregated_results.csv"
data = pd.read_csv(data_path)
data.head()

,file,scenario,git_info,max_iter,LLM,LLM_temp,NOFEEDBACK,GENCMD,RUNFUZZ,subject,commit,final_result,unintended_bug,time,iid,score,success
0,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,100,5013,1,0
1,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,D,False,97,5138,2,0
2,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,53b61c1,D,False,57,5117,2,0
3,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,B,False,36,5153,3,1
4,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,107,5013,1,0


In [2]:
# check the unique values in the column
for column in data.columns:
    if column in ["file", "time"]:
        continue
    print(f"{column}: {data[column].unique()}")

ALL_SUBJECTS = ['mujs', 'libxml2', 'poppler',    # submission v1
                'jerryscript', 'z3', 'php-src',  # submission v2
                'jq', 'micropython']             # submission v3.5
ALL_SUBJECTS_LATEX = {
    "mujs": r"\mujs",
    "libxml2": r"\libxml",
    "poppler": r"\poppler",
    "jerryscript": r"\jerryscript",
    "z3": r"\zthree",
    "php-src": r"\php",
    "jq": r"\jq",
    "micropython": r"\micropython"
}

# get number of total iids
N_IID = data['iid'].nunique()
N_DATAPOINT = N_IID * 20  # repeat 10 times  * 2 scenarios for each iid per setting
print(f"Total number of bug issues (iids): {N_IID}")
print(f"Each setting should have {N_DATAPOINT} data points")

scenario: <StringArray>
['BIC', 'FIX']
Length: 2, dtype: str
git_info: <StringArray>
['DIFFONLY', 'FULL', 'MSGONLY', 'ENHANCED', 'REDUCED']
Length: 5, dtype: str
max_iter: [ 5 10]
LLM: <StringArray>
['gpt-4o-2024-08-06', 'deepseek-r1', 'gpt-4o-mini']
Length: 3, dtype: str
LLM_temp: [0.5 1. ]
NOFEEDBACK: [nan  1.]
GENCMD: [nan  1.]
RUNFUZZ: [nan]
subject: <StringArray>
['jerryscript',          'jq',     'libxml2', 'micropython',        'mujs',
     'php-src',     'poppler',          'z3']
Length: 8, dtype: str
commit: <StringArray>
['b7e3bae', '70e275e', '53b61c1', '3686410', '95cc5e9', 'd7e2125', 'de51531',
 '680baef', '0e067ef', '4003202', '3fa10e8', '71c2ab5', 'de21386', '499c91b',
 '9a82b94', '74c84a8', '1406b20', '14604a4', '9cfc723', 'd0c3f01', '6273df6',
 'db21cd5', '3ad7f81', 'dc6270d', '4c93955', '1fef566', '5e9189d', '69ead7d',
 'e702046', '7e14680', 'f397a3e', '908ab1c', '3185bb5', 'c0252d7', '4b013ec',
 '0615d13', '8c27b12', '832e069', '4c7f6be', '3f71a1c', '833f82c', '6871e

In [3]:
# get results for a specific config, like MSGONLY
data_msgonly = data[data['git_info'] == 'MSGONLY']
# count success of msgonly
data_msgonly_success = data_msgonly[data_msgonly['success'] == 1]
# unique by commit
data_msgonly_success_unique = data_msgonly_success.drop_duplicates(subset=['subject', 'commit'])
# group by scenario
data_msgonly_success_unique = data_msgonly_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_msgonly_success_unique)

# get results where final_result is 'X', need to manually inspect reachability and score
data_x = data[(data['final_result'] == 'X') & (data['git_info'] == 'FULL') & (data['max_iter'] == 5) & (data['LLM'] == 'gpt-4o-2024-08-06')]
# unique on iid and scenario, sorted by iid
data_x = data_x.drop_duplicates(subset=['subject', 'commit', 'iid'])
data_x = data_x.sort_values(by=['iid'])
print(data_x.to_csv())

,scenario,count
0,BIC,7
1,FIX,13


,file,scenario,git_info,max_iter,LLM,LLM_temp,NOFEEDBACK,GENCMD,RUNFUZZ,subject,commit,final_result,unintended_bug,time,iid,score,success
3721,SUMMARY_mujs_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_250905-2232.txt,BIC,FULL,5,gpt-4o-2024-08-06,0.5,,,,mujs,832e069,X,True,46,141,2,0
2221,SUMMARY_libxml2_FIX_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_GENCMD_260203-0916.txt,FIX,FULL,5,gpt-4o-2024-08-06,0.5,,1.0,,libxml2,6273df6,X,True,44,550,0,0
5191,SUMMARY_poppler_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_250827-1940.txt,BIC,FULL,5,gpt-4o-2024-08-06,0.5,,,,poppler,3cae777,X,True,126,1289,0,0
128,SUMMARY_jerryscript_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_250826-2145.txt,BIC,FULL,5,gpt-4o-2024-08-06,0.5,,,,jerryscript,b7e3bae,X,True,78,5013,1,0
1259,SUMMARY_jq_FIX_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_260117-0915.txt,FIX,FULL,5,gpt-4o-2024-08-06,0.5,,,,jq,499c91b,X,True,47,49014,0,0
867,SUMMARY_jq_BIC_GITFULL_ITER5_gpt-4o-2024-08-06_TEMP0.5_260117-0919.txt,BIC,FULL,5,gpt-4o-2024-08-

In [4]:
try:
    subject_order = pd.read_csv("targets.csv")["software"].unique().tolist()
except:
    subject_order = ["mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython"]
print(f"Subject order: {subject_order}")

data["subject"] = pd.Categorical(data["subject"], categories=subject_order, ordered=True)

Subject order: ['mujs', 'libxml2', 'poppler', 'jerryscript', 'z3', 'php-src', 'jq', 'micropython']


### Table 2
Results for bug finding and bug reproduction effectiveness of Cleverest (default configuration)

In [5]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# check the number of datapoints; expected to be 720 = 36 iid * 2 scenario * 10 repetition; if not, print warning
if len(data_def) != N_DATAPOINT:
    print(
        f"!!Warning!!: the number of datapoints is {len(data_def)}; expected to be {N_DATAPOINT}"
    )
data_def["reached"] = data_def["score"].apply(lambda x: 1 if x >= 1 else 0)
data_def["changed"] = data_def["score"].apply(lambda x: 1 if x >= 2 else 0)
# aggregate the data regarding the scenario and iid
data_def = (
    data_def.groupby(["scenario", "subject", "iid"])
    # .agg({"score": "mean", "success": "sum", "time": "mean"})
    .agg({"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"})
    .reset_index()
)
# scenario to multi columns, scenario first then (score, success, time)
data_def_pivot = data_def.pivot(index=["subject", "iid"], columns="scenario")
# order the column index by Subject (level 0) first, then IID (level 1)
data_def_pivot = data_def_pivot.sort_index(level=[0, 1])
# add average row
average = data_def_pivot.agg({
    ("score", "BIC"): "mean",
    ("score", "FIX"): "mean",
    ("time", "BIC"): "mean",
    ("time", "FIX"): "mean",
    ("reached", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("reached", "FIX"): lambda x: (x != 0).sum(),
    ("changed", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("changed", "FIX"): lambda x: (x != 0).sum(),
    ("success", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("success", "FIX"): lambda x: (x != 0).sum(),
}).to_frame().T
average.index = pd.MultiIndex.from_tuples(
    [(r"\multicolumn{2}{c|}{Average}", "")]
)
data_def_pivot = pd.concat([data_def_pivot, average])
# check the column types
# display(data_def_pivot)
# sort data_def by scenario, and then iid
# Define subject order from targets.csv to ensure correct grouping (based on targets.csv order)
try:
    subject_order = pd.read_csv("targets.csv")["software"].unique().tolist()
except:
    subject_order = ["mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython"]

data_def["subject"] = pd.Categorical(data_def["subject"], categories=subject_order, ordered=True)

# sort data_def by subject, iid, and scenario
data_def = data_def.sort_values(by=["subject", "iid", "scenario"])
display(data_def)

,scenario,subject,iid,score,reached,changed,success,time
0,BIC,mujs,65,3.0,10,10,10,7.5
36,FIX,mujs,65,2.2,10,6,6,19.1
1,BIC,mujs,141,2.0,10,10,0,45.4
37,FIX,mujs,141,2.1,10,10,1,31.7
2,BIC,mujs,145,2.6,10,8,8,37.1
...,...,...,...,...,...,...,...,...
69,FIX,micropython,17815,3.0,10,10,10,5.5
34,BIC,micropython,17841,2.0,10,10,0,21.4
70,FIX,micropython,17841,1.0,10,0,0,39.2
35,BIC,micropython,17847,2.0,10,10,0,40.9


Formatting for the latex table

In [6]:
# order: iid
data_def_pivot_latex = data_def_pivot.copy()

###### index formats
# mujs -> \mujs
# libxml2 -> \libxml
# poppler -> \poppler
# jerryscript -> \jerryscript
# php-src -> \php
# z3 -> \zthree
##### also the level 1 should be "3" -> "\#3"
data_def_pivot_latex.index = data_def_pivot_latex.index.set_levels(
    [
        data_def_pivot_latex.index.levels[0].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "micropython": r"\micropython",
                "jq": r"\jq",
                r"\multicolumn{2}{c|}{Average}": r"\multicolumn{2}{c|}{Average}",
            }
        ),
        data_def_pivot_latex.index.levels[1].map(lambda x: f"\\#{x}" if x else ""),
    ]
)

###### column formats
# score -> \SLIDER{3.3em}{score / 3:.2f}{circle}
data_def_pivot_latex["score"] = data_def_pivot_latex["score"].map(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
data_def_pivot_latex["reached"] = data_def_pivot_latex["reached"].map(
    lambda x: f"{int(x)}/10"
)
data_def_pivot_latex["changed"] = data_def_pivot_latex["changed"].map(
    lambda x: f"{int(x)}/10"
)
# success -> {success}/10
data_def_pivot_latex["success"] = data_def_pivot_latex["success"].map(
    lambda x: f"{int(x)}/10"
)
# time -> {time:.1f}
data_def_pivot_latex["time"] = data_def_pivot_latex["time"].map(
    lambda x: f"{x:.1f}"
)

# Fix Average row denominator: X/10 -> X/{N_IID}
avg_idx = (r"\multicolumn{2}{c|}{Average}", "")
for col in ["reached", "changed", "success"]:
    data_def_pivot_latex.loc[avg_idx, col] = data_def_pivot.loc[avg_idx, col].apply(lambda x: f"{int(x)}/{N_IID}").values

# swap the column level
data_def_pivot_latex = data_def_pivot_latex.swaplevel(axis=1)
# sort the column level
data_def_pivot_latex = data_def_pivot_latex.sort_index(axis=1)
# change column order for to (BIC FIX) x (score reached changed success time)
data_def_pivot_latex = data_def_pivot_latex.reindex(
    columns=pd.MultiIndex.from_product(
        [["BIC", "FIX"], ["score", "reached", "changed", "success", "time"]]
    )
)

# show the columns
display(data_def_pivot_latex)


latex_str = data_def_pivot_latex.to_latex()
# remove [t] from multirow
latex_str = latex_str.replace("[t]", "").replace(
    r"\multicolumn{2}{c|}{Average} &  &",
    r"\multicolumn{2}{c|}{Average} &"
).replace(
    r"\cline{1-8}", r"\midrule"
).replace(
    r"\cline{1-12}", r"\midrule"
)



print(latex_str)
# print(data_def_pivot_latex.to_latex())

BIC          \
                                                             score reached   
\mujs                        \#65     \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#141    \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#145    \SLIDER{3.3em}{0.87}{circle}   10/10   
                             \#166    \SLIDER{3.3em}{0.37}{circle}   10/10   
\libxml                      \#535    \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#550    \SLIDER{3.3em}{0.40}{circle}   10/10   
                             \#553    \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#720    \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#841    \SLIDER{3.3em}{1.00}{circle}   10/10   
\poppler                     \#1282   \SLIDER{3.3em}{0.37}{circle}    8/10   
                             \#1289   \SLIDER{3.3em}{0.13}{circle}    4/10   
                             \#1303   \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#1305   \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#1381   \SLIDER{3.3em}{0.00}{circle}    0/10   
\jerryscript                 \#5013   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#5117   \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#5138   \SLIDER{3.3em}{0.73}{circle}   10/10   
                             \#5153   \SLIDER{3.3em}{0.67}{circle}   10/10   
\zthree                      \#6659   \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#6914   \SLIDER{3.3em}{0.07}{circle}    2/10   
                             \#7246   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#7252   \SLIDER{3.3em}{0.27}{circle}    8/10   
\php                         \#16777  \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#16978  \SLIDER{3.3em}{0.37}{circle}   10/10   
                             \#17442  \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#17463  \SLIDER{3.3em}{0.20}{circle}    6/10   
\jq                          \#2825   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#2976   \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#3262   \SLIDER{3.3em}{1.00}{circle}   10/10   
                             \#49014  \SLIDER{3.3em}{0.13}{circle}    2/10   
\micropython                 \#13007  \SLIDER{3.3em}{0.53}{circle}    9/10   
                             \#13041  \SLIDER{3.3em}{0.00}{circle}    0/10   
                             \#17733  \SLIDER{3.3em}{0.33}{circle}   10/10   
                             \#17815  \SLIDER{3.3em}{0.63}{circle}   10/10   
                             \#17841  \SLIDER{3.3em}{0.67}{circle}   10/10   
                             \#17847  \SLIDER{3.3em}{0.67}{circle}   10/10   
\multicolumn{2}{c|}{Average}          \SLIDER{3.3em}{0.46}{circle}   31/36   

                                                             \
                                     changed success   time   
\mujs                        \#65      10/10   10/10    7.5   
                             \#141     10/10    0/10   45.4   
                             \#145      8/10    8/10   37.1   
                             \#166      1/10    0/10   57.6   
\libxml                      \#535     10/10   10/10    5.4   
                             \#550      2/10    0/10   27.6   
                             \#553     10/10   10/10    6.7   
                             \#720      0/10    0/10   53.6   
                             \#841     10/10   10/10    5.9   
\poppler                     \#1282     3/10    0/10  196.6   
                             \#1289     0/10    0/10  165.0   
                             \#1303     0/10    0/10  218.7   
                             \#1305     0/10    0/10  267.2   
                             \#1381     0/10    0/10  202.8   
\jerryscript

\begin{tabular}{llllllllllll}
\toprule
 &  & \multicolumn{5}{r}{BIC} & \multicolumn{5}{r}{FIX} \\
 &  & score & reached & changed & success & time & score & reached & changed & success & time \\
\midrule
\multirow{4}{*}{\mujs} & \#65 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 7.5 & \SLIDER{3.3em}{0.73}{circle} & 10/10 & 6/10 & 6/10 & 19.1 \\
 & \#141 & \SLIDER{3.3em}{0.67}{circle} & 10/10 & 10/10 & 0/10 & 45.4 & \SLIDER{3.3em}{0.70}{circle} & 10/10 & 10/10 & 1/10 & 31.7 \\
 & \#145 & \SLIDER{3.3em}{0.87}{circle} & 10/10 & 8/10 & 8/10 & 37.1 & \SLIDER{3.3em}{0.93}{circle} & 10/10 & 9/10 & 9/10 & 28.8 \\
 & \#166 & \SLIDER{3.3em}{0.37}{circle} & 10/10 & 1/10 & 0/10 & 57.6 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 7.4 \\
\midrule
\multirow{5}{*}{\libxml} & \#535 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 5.4 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 5.5 \\
 & \#550 & \SLIDER{3.3em}{0.40}{circle} & 10/10 & 2/10 & 0/10 & 27.6 & \SLID

In [7]:
# NOTE: RQ1 group subjects into good/bad for discussion
# 1. Calculate BIC and FIX averages for each subject
subject_avg_scores = data_def.groupby(['subject', 'scenario'], observed=True)['score'].mean().unstack()

# 2. Add a column for the combined average of BIC and FIX
subject_avg_scores['Combined Avg'] = subject_avg_scores[['BIC', 'FIX']].mean(axis=1)

# 3. Sort the subjects by the Combined Avg in descending order
subject_avg_scores = subject_avg_scores.sort_values(by='Combined Avg', ascending=False)

# 4. Calculate and append the Overall Average row at the bottom
subject_avg_scores.loc['Overall Average'] = subject_avg_scores.mean()

print("Subject Average Scores (0-3), Sorted by Combined Average:")
print(subject_avg_scores)

Subject Average Scores (0-3), Sorted by Combined Average:
scenario              BIC       FIX  Combined Avg
subject                                          
mujs             2.175000  2.525000      2.350000
libxml2          2.240000  1.940000      2.090000
jq               1.350000  1.775000      1.562500
jerryscript      1.800000  1.000000      1.400000
micropython      1.416667  1.200000      1.308333
php-src          1.175000  1.000000      1.087500
poppler          0.300000  0.620000      0.460000
z3               0.500000  0.125000      0.312500
Overall Average  1.369583  1.273125      1.321354


In [8]:
# Filter for the specific subjects mentioned in the text
good_subjects = ["mujs", "libxml2", "jerryscript", "jq"]
soso_subjects = ["php-src", "micropython"]
bad_subjects = ["poppler", "z3"]
subset_good = data_def[data_def["subject"].isin(good_subjects)]

print(f"--- Analysis for good subjects: {good_subjects} ---")

# 1. Bug finding (BIC) success: "identified bugs in 4 out of 10 BICs"
bic_success = subset_good[subset_good["scenario"] == "BIC"]["success"].apply(lambda x: 1 if x > 0 else 0).sum()
bic_total = subset_good[subset_good["scenario"] == "BIC"]["iid"].nunique()
print(f"BICs found: {bic_success} out of {bic_total}")
# 2. Bug reproduction (FIX/BFC) success: "reproduced bugs patched in 6 out of 10 BFCs"
bfc_success = subset_good[subset_good["scenario"] == "FIX"]["success"].apply(lambda x: 1 if x > 0 else 0).sum()
bfc_total = len(subset_good[subset_good["scenario"] == "FIX"])
print(f"BFCs reproduced: {bfc_success} out of {bfc_total}")
# 3. Reaching changed code: "remaining 8 out of 10 commits... at least reached"
# 'Remaining' implies (Total - Successes), and we check if 'reached' > 0
remaining_bic_bfc = subset_good[subset_good["success"] == 0]
reached_in_remaining = remaining_bic_bfc["reached"].apply(lambda x: 1 if x > 0 else 0).sum()
print(f"Reached in remaining commits: {reached_in_remaining} out of {len(remaining_bic_bfc)}")

print(f"\n--- Analysis for soso subjects: {soso_subjects} ---")
subset_soso = data_def[data_def["subject"].isin(soso_subjects)]
success_soso = subset_soso["success"].apply(lambda x: 1 if x > 0 else 0).sum()
total_soso = subset_soso["iid"].nunique() * 2  # two commits per iid (BIC and FIX)
print(f"Bug found in soso subjects: {success_soso} out of {total_soso}")
remaining_soso = subset_soso[subset_soso["success"] == 0]
reached_in_remaining_soso = remaining_soso["reached"].apply(lambda x: 1 if x > 0 else 0).sum()
print(f"Reached in remaining soso commits: {reached_in_remaining_soso} out of {len(remaining_soso)}")

print(f"\n--- Analysis for bad subjects: {bad_subjects} ---")
subset_bad = data_def[data_def["subject"].isin(bad_subjects)]
success_bad = subset_bad["success"].apply(lambda x: 1 if x > 0 else 0).sum()
total_bad = subset_bad["iid"].nunique() * 2  # two commits per iid (BIC and FIX)
print(f"Bug found in bad subjects: {success_bad} out of {total_bad}")
remaining_bad = subset_bad[subset_bad["success"] == 0]
reached_in_remaining_bad = remaining_bad["reached"].apply(lambda x: 1 if x > 0 else 0).sum()
print(f"Reached in remaining bad commits: {reached_in_remaining_bad} out of {len(remaining_bad)}")

# subset_complex = data_def[data_def["subject"].isin(["php-src", "z3"])]
# # "Found a bug in a BFC" (Success > 0 in FIX scenario)
# php_z3_bfc_found = subset_complex[(subset_complex["scenario"] == "FIX") & (subset_complex["success"] > 0)]
# print(f"BFC bugs found in PHP/Z3: {len(php_z3_bfc_found)}")
# # "Reached the changed code in 8 out of 11 remaining cases"
# remaining_complex = subset_complex[subset_complex["success"] == 0]
# reached_complex = remaining_complex["reached"].apply(lambda x: 1 if x > 0 else 0).sum()
# print(f"Reached in remaining PHP/Z3 cases: {reached_complex} out of {len(remaining_complex)}")

--- Analysis for good subjects: ['mujs', 'libxml2', 'jerryscript', 'jq'] ---
BICs found: 7 out of 17
BFCs reproduced: 10 out of 17
Reached in remaining commits: 14 out of 17

--- Analysis for soso subjects: ['php-src', 'micropython'] ---
Bug found in soso subjects: 2 out of 20
Reached in remaining soso commits: 14 out of 18

--- Analysis for bad subjects: ['poppler', 'z3'] ---
Bug found in bad subjects: 1 out of 18
Reached in remaining bad commits: 8 out of 17


### Table 3
Results for our ablation study with average effectiveness score on a slider.

In [9]:
config_prompt_msg = config_def.copy()
config_prompt_msg["git_info"] = "MSGONLY"
config_prompt_diff = config_def.copy()
config_prompt_diff["git_info"] = "DIFFONLY"
config_prompt_gen = config_def.copy()
config_prompt_gen["GENCMD"] = 1
config_llm_temp1 = config_def.copy()
config_llm_temp1["LLM_temp"] = 1
config_llm_mini = config_def.copy()
config_llm_mini["LLM"] = "gpt-4o-mini"
config_llm_deepseek = config_def.copy()
config_llm_deepseek["LLM"] = "deepseek-r1"
config_exec_iter10 = config_def.copy()
config_exec_iter10["max_iter"] = 10
config_exec_xfeedback = config_def.copy()
config_exec_xfeedback["NOFEEDBACK"] = 1

configs = {
    "default-": config_def,
    "prompt-msg": config_prompt_msg,
    "prompt-diff": config_prompt_diff,
    "prompt-gen": config_prompt_gen,
    "llm-temp1": config_llm_temp1,
    "llm-mini": config_llm_mini,
    "llm-deepseek": config_llm_deepseek,
    "exec-iter10": config_exec_iter10,
    "exec-xfeedback": config_exec_xfeedback,
}

data_each_config = []
for config_key, config in configs.items():
    data_config = data.copy()
    for key, value in config.items():
        if pd.isna(value):
            data_config = data_config[pd.isna(data_config[key])]
        else:
            data_config = data_config[data_config[key] == value]
    # check the number of datapoints for each config
    print(f"{config_key}: {len(data_config)}")
    # 66 commits in total, if len(data_config) != 660, print warning
    if len(data_config) != N_DATAPOINT:
        if config_key == "prompt-gen" and len(data_config) == 220:
            pass  # only 2+5+4 bugs for libxml2,poppler,jq need GENCMD, 11*10*2=220
        else:
            print(
                f"!!Warning!! {config_key} has {len(data_config)} data points; expected {N_DATAPOINT}"
            )
    data_config = (
        data_config.groupby(["scenario", "subject", "iid"])
        .agg({"score": "mean"})
        .reset_index()
    )
    data_config["config_category"] = config_key.split("-")[0]
    data_config["config"] = config_key.split("-")[1]
    data_each_config.append(data_config)

data_each_config = pd.concat(data_each_config)
data_each_config = data_each_config.pivot(
    index=["scenario", "subject", "iid"],
    columns=["config_category", "config"],
    values="score",
)
# for the column ("prompt", "gen"), if the value is NaN, fill it with the value in ("default", "default")
data_each_config[("prompt", "gen")] = data_each_config[("prompt", "gen")].fillna(
    data_each_config[("default", "")]
)

# NOTE: FIX poppler 1305 (prompt, msg) data is wrong, if it is 0.2, manually change to 0.0
if data_each_config.loc[("FIX", "poppler", 1305), ("prompt", "msg")] == 0.2:
    data_each_config.loc[("FIX", "poppler", 1305), ("prompt", "msg")] = 0.0

# NOTE: RQ2 main text
# gpt-4o-mini worse than default
data_gpt4omini_worse = data_each_config[data_each_config[("llm", "mini")] < data_each_config[("default", "")]]
display(data_gpt4omini_worse)
print(f"# of data gpt-4o-mini is worse than default: {len(data_gpt4omini_worse)}")
data_gpt4omini_worse_unreach = data_gpt4omini_worse[data_gpt4omini_worse[("llm", "mini")] == 0]
display(data_gpt4omini_worse_unreach)
print(f"# of data where gpt-4o-mini is worse and also unreachable: {len(data_gpt4omini_worse_unreach)}")

# iter10 better than default
data_iter10_better = data_each_config[data_each_config[("exec", "iter10")] < data_each_config[("default", "")]]
display(data_iter10_better)
print(f"# of data where iter10 is better than default: {len(data_iter10_better)}")

# nofeedback worse than default
data_nofeedback_worse = data_each_config[data_each_config[("exec", "xfeedback")] < data_each_config[("default", "")]]
display(data_nofeedback_worse)
print(f"# of data where xfeedback is worse than default: {len(data_nofeedback_worse)}")
data_nofeedback_worse_unreach = data_nofeedback_worse[data_nofeedback_worse[("exec", "xfeedback")] == 0]
display(data_nofeedback_worse_unreach)
print(f"# of data where xfeedback is worse and also unreachable: {len(data_nofeedback_worse_unreach)}")

# gencmd 
data_gencmd_worse = data_each_config[data_each_config[("prompt", "gen")] + 1 <= data_each_config[("default", "")]]
display(data_gencmd_worse)
print(f"# of data where gencmd is worse than default: {len(data_gencmd_worse)}")
data_gencmd_worse_unreach = data_gencmd_worse[data_gencmd_worse[("prompt", "gen")] == 0]
display(data_gencmd_worse_unreach)
print(f"# of data where gencmd is worse and also unreachable: {len(data_gencmd_worse_unreach)}")

# Unstack scenario from index to columns (BIC | FIX layout)
data_each_config = data_each_config.unstack(level=0)
# After unstack, columns are: (config_category, config, scenario)
# Reorder to: (scenario, config_category, config)
data_each_config.columns = data_each_config.columns.reorder_levels([2, 0, 1])

# Reorder columns to match configs dict order: default, prompt (msg, diff, gen), llm (temp1, mini, deepseek), exec (iter10, xfeedback)
scenario_order = ["BIC", "FIX"]
config_category_order = ["default", "prompt", "llm", "exec"]
config_order = {"default": [""], "prompt": ["msg", "diff", "gen"], "llm": ["temp1", "mini", "deepseek"], "exec": ["iter10", "xfeedback"]}

# Build the desired column order
desired_columns = []
for scenario in scenario_order:
    for category in config_category_order:
        for cfg in config_order[category]:
            desired_columns.append((scenario, category, cfg))

data_each_config = data_each_config[desired_columns]

# reorder the index level 0: ["mujs", "libxml2", "poppler", "jerryscript", "php-src", "z3", "jq", "micropython"]
data_each_config.index = pd.MultiIndex.from_arrays([
    pd.Categorical(data_each_config.index.get_level_values(0), categories=[
        "mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython"
    ], ordered=True),
    data_each_config.index.get_level_values(1)
])
data_each_config = data_each_config.sort_index()

# add average row
average = data_each_config.agg("mean").to_frame().T
average.index = pd.MultiIndex.from_tuples([("Average", "")])
data_each_config = pd.concat([data_each_config, average])

data_each_config

default-: 720
prompt-msg: 720
prompt-diff: 720
prompt-gen: 280
!!Warning!! prompt-gen has 280 data points; expected 720
llm-temp1: 720
llm-mini: 720
llm-deepseek: 720
exec-iter10: 720
exec-xfeedback: 720


config_category            default prompt             llm                \
config                                msg diff  gen temp1 mini deepseek   
scenario subject     iid                                                  
BIC      mujs        65        3.0    3.0  3.0  3.0   3.0  0.6      3.0   
                     145       2.6    3.0  2.6  2.6   2.6  1.2      3.0   
                     166       1.1    1.3  1.0  1.1   1.2  1.0      1.8   
         poppler     1282      1.1    1.3  1.0  1.0   1.4  0.1      1.6   
         jerryscript 5013      1.0    1.0  1.0  1.0   1.1  0.1      1.7   
                     5138      2.2    2.0  2.3  2.2   2.2  2.0      2.0   
         z3          7252      0.8    0.0  0.9  0.8   0.4  0.0      1.0   
         php-src     16777     1.0    0.6  0.9  1.0   1.0  0.9      1.0   
                     16978     1.1    1.5  1.3  1.1   1.3  1.0      1.2   
                     17442     2.0    1.7  2.0  2.0   1.8  1.8      2.0   
                     17463     0.6    1.0  0.3  0.6   0.7  0.3      0.9   
         jq          3262      3.0    3.0  3.0  1.1   3.0  0.0      3.0   
                     49014     0.4    0.0  0.4  0.2   0.6  0.0      1.3   
         micropython 13007     1.6    2.3  1.1  1.6   1.5  0.0      1.5   
                     17841     2.0    2.0  2.0  2.0   2.3  1.9      2.0   
                     17847     2.0    2.0  2.0  2.0   2.0  0.0      3.0   
FIX      mujs        65        2.2    1.0  2.2  2.2   2.2  1.0      2.7   
                     145       2.8    2.4  2.7  2.8   2.0  0.8      3.0   
         poppler     1282      0.9    1.0  0.9  1.0   1.0  0.3      0.2   
                     1289      2.0    0.0  1.5  0.0   2.1  0.0      3.0   
                     1303      0.2    0.0  0.7  0.0   1.2  0.0      1.8   
         jerryscript 5117      2.0    1.4  1.0  2.0   2.2  1.0      1.6   
                     5138      1.0    1.0  1.0  1.0   1.6  0.5      3.0   
         z3          7252      0.5    0.0  0.6  0.5   0.4  0.0      1.0   
         php-src     16777     1.0    0.6  0.8  1.0   0.9  0.5      1.0   
         jq          2825      1.1    1.0  1.3  1.4   1.2  1.0      1.2   
                     3262      3.0    2.8  3.0  1.1   3.0  2.0      3.0   
         micropython 13007     0.2    0.5  0.7  0.2   0.0  0.0      3.0   

config_category              exec            
config                     iter10 xfeedback  
scenario subject     iid                     
BIC      mujs        65       3.0       2.7  
                     145      3.0       2.2  
                     166      1.1       1.1  
         poppler     1282     1.2       1.1  
         jerryscript 5013     1.0       1.0  
                     5138     2.4       2.3  
         z3          7252     0.7       0.1  
         php-src     16777    1.0       1.0  
                     16978    1.3       1.2  
                     17442    2.0       2.0  
                     17463    0.8       0.0  
         jq          3262     3.0       0.3  
                     49014    0.5       0.2  
         micropython 13007    1.9       0.0  
                     17841    2.1       2.2  
                     17847    2.0       2.0  
FIX      mujs        65       2.2       1.4  
                     145      3.0       2.8  
         poppler     1282     1.0       0.9  
                     1289     2.6       2.0  
                     1303     1.0       0.0  
         jerryscript 5117     1.6       2.0  
                     5138     1.4       1.5  
         z3          7252     0.8       0.2  
         php-src     16777    0.9       0.3  
         jq          2825     1.1       1.0  
                     3262     3.0       3.0  
         micropython 13007    0.3       0.0

# of data gpt-4o-mini is worse than default: 28


config_category            default prompt             llm                \
config                                msg diff  gen temp1 mini deepseek   
scenario subject     iid                                                  
BIC      z3          7252      0.8    0.0  0.9  0.8   0.4  0.0      1.0   
         jq          3262      3.0    3.0  3.0  1.1   3.0  0.0      3.0   
                     49014     0.4    0.0  0.4  0.2   0.6  0.0      1.3   
         micropython 13007     1.6    2.3  1.1  1.6   1.5  0.0      1.5   
                     17847     2.0    2.0  2.0  2.0   2.0  0.0      3.0   
FIX      poppler     1289      2.0    0.0  1.5  0.0   2.1  0.0      3.0   
                     1303      0.2    0.0  0.7  0.0   1.2  0.0      1.8   
         z3          7252      0.5    0.0  0.6  0.5   0.4  0.0      1.0   
         micropython 13007     0.2    0.5  0.7  0.2   0.0  0.0      3.0   

config_category              exec            
config                     iter10 xfeedback  
scenario subject     iid                     
BIC      z3          7252     0.7       0.1  
         jq          3262     3.0       0.3  
                     49014    0.5       0.2  
         micropython 13007    1.9       0.0  
                     17847    2.0       2.0  
FIX      poppler     1289     2.6       2.0  
                     1303     1.0       0.0  
         z3          7252     0.8       0.2  
         micropython 13007    0.3       0.0

# of data where gpt-4o-mini is worse and also unreachable: 9


...

config_category       default prompt             llm                 exec  \
config                           msg diff  gen temp1 mini deepseek iter10   
scenario subject iid                                                        
BIC      libxml2 535      3.0    3.0  3.0  1.2   3.0  3.0      3.0    3.0   
                 553      3.0    3.0  3.0  1.0   3.0  3.0      3.0    3.0   
                 841      3.0    3.0  3.0  2.0   3.0  3.0      3.0    3.0   
         jq      3262     3.0    3.0  3.0  1.1   3.0  0.0      3.0    3.0   
FIX      libxml2 553      3.0    3.0  3.0  0.4   3.0  3.0      3.0    3.0   
         poppler 1289     2.0    0.0  1.5  0.0   2.1  0.0      3.0    2.6   
         jq      3262     3.0    2.8  3.0  1.1   3.0  2.0      3.0    3.0   

config_category                  
config                xfeedback  
scenario subject iid             
BIC      libxml2 535        3.0  
                 553        3.0  
                 841        3.0  
         jq      3262       0.3  
FIX      libxml2 553        3.0  
         poppler 1289       2.0  
         jq      3262       3.0

# of data where gencmd is worse than default: 7


config_category       default prompt             llm                 exec  \
config                           msg diff  gen temp1 mini deepseek iter10   
scenario subject iid                                                        
FIX      poppler 1289     2.0    0.0  1.5  0.0   2.1  0.0      3.0    2.6   

config_category                  
config                xfeedback  
scenario subject iid             
FIX      poppler 1289       2.0

# of data where gencmd is worse and also unreachable: 1


scenario                BIC                                                    \
config_category     default    prompt                           llm             
config                            msg      diff       gen     temp1      mini   
mujs        65     3.000000  3.000000  3.000000  3.000000  3.000000  0.600000   
            141    2.000000  2.000000  2.000000  2.000000  2.000000  2.000000   
            145    2.600000  3.000000  2.600000  2.600000  2.600000  1.200000   
            166    1.100000  1.300000  1.000000  1.100000  1.200000  1.000000   
libxml2     535    3.000000  3.000000  3.000000  1.200000  3.000000  3.000000   
            550    1.200000  1.900000  1.100000  1.200000  1.500000  1.500000   
            553    3.000000  3.000000  3.000000  1.000000  3.000000  3.000000   
            720    1.000000  1.000000  1.000000  1.100000  1.000000  1.000000   
            841    3.000000  3.000000  3.000000  2.000000  3.000000  3.000000   
poppler     1282   1.100000  1.300000  1.000000  1.000000  1.400000  0.100000   
            1289   0.400000  0.600000  1.000000  0.900000  1.500000  0.400000   
            1303   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
            1305   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
            1381   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
jerryscript 5013   1.000000  1.000000  1.000000  1.000000  1.100000  0.100000   
            5117   2.000000  2.000000  2.000000  2.000000  2.000000  2.000000   
            5138   2.200000  2.000000  2.300000  2.200000  2.200000  2.000000   
            5153   2.000000  2.000000  2.300000  2.000000  2.100000  2.000000   
z3          6659   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
            6914   0.200000  0.100000  0.400000  0.200000  0.400000  0.200000   
            7246   1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
            7252   0.800000  0.000000  0.900000  0.800000  0.400000  0.000000   
php-src     16777  1.000000  0.600000  0.900000  1.000000  1.000000  0.900000   
            16978  1.100000  1.500000  1.300000  1.100000  1.300000  1.000000   
            17442  2.000000  1.700000  2.000000  2.000000  1.800000  1.800000   
            17463  0.600000  1.000000  0.300000  0.600000  0.700000  0.300000   
jq          2825   1.000000  1.200000  1.000000  1.000000  1.000000  1.000000   
            2976   1.000000  1.000000  1.000000  1.100000  1.000000  1.000000   
            3262   3.000000  3.000000  3.000000  1.100000  3.000000  0.000000   
            49014  0.400000  0.000000  0.400000  0.200000  0.600000  0.000000   
micropython 13007  1.600000  2.300000  1.100000  1.600000  1.500000  0.000000   
            13041  0.000000  0.300000  0.300000  0.000000  0.000000  0.000000   
            17733  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
            17815  1.900000  1.800000  1.100000  1.900000  2.000000  1.900000   
            17841  2.000000  2.000000  2.000000  2.000000  2.300000  1.900000   
            17847  2.000000  2.000000  2.000000  2.000000  2.000000  0.000000   
Average            1.366667  1.405556  1.361111  1.191667  1.433333  0.969444   

scenario                                              FIX                  \
config_category                  exec             default    prompt         
config             deepseek    iter10 xfeedback                 msg  diff   
mujs        65     3.000000  3.000000  2.700000  2.200000  1.000000  2.20   
            141    2.000000  2.000000  2.000000  2.100000  0.000000  1.70   
            145    3.000000  3.000000  2.200000  2.800000  2.400000  2.70   
            166    1.800000  1.100000  1.100000  3.000000  3.000000  2.70   
libxml2     535    3.000000  3.000000  3.000000  3.000000  3.000000  3.00   
            550    2.000000  1.400000  1.100000  0.000000  0.000000  0.00   
            553    3.000000  3.000000  3.000000  3.000000  3.00000

In [10]:
# to latex
# Store raw values BEFORE any transformations
data_raw_values = data_each_config.copy()

data_each_config_latex = data_each_config.astype(object).copy()  # Convert to object dtype to allow strings

# index formats
data_each_config_latex.index = data_each_config_latex.index.set_levels(
    [
        data_each_config_latex.index.levels[0].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "jq": r"\jq",
                "micropython": r"\micropython",
                "Average": r"\multicolumn{2}{c||}{Average}",
            }
        ),
        data_each_config_latex.index.levels[1].map(
            lambda x: f"\\#{x}" if x else ""
        ),
    ]
)

###### column formats
# Format each cell: regular rows get slider, Average row gets slider + raw value below
# Need to use the original (unformatted) index for data_raw_values
for i, idx_formatted in enumerate(data_each_config_latex.index):
    idx_raw = data_raw_values.index[i]  # Get original index
    is_avg = idx_raw[0] == "Average"  # Check if this is the Average row
    for col in data_each_config_latex.columns:
        raw_value = data_raw_values.loc[idx_raw, col]
        if pd.notna(raw_value):
            scaled = raw_value / 3
            if is_avg:
                # Average row: nested tabular with slider and raw value
                data_each_config_latex.loc[idx_formatted, col] = f"\\begin{{tabular}}{{c}}\\SLIDER{{\\mysliderlen}}{{{scaled:.2f}}}{{circle}}\\vspace{{-0.1cm}}\\\\\\small{{{raw_value:.2f}}}\\end{{tabular}}"
            else:
                # Regular rows: just the slider
                data_each_config_latex.loc[idx_formatted, col] = f"\\SLIDER{{\\mysliderlen}}{{{scaled:.2f}}}{{circle}}"

display(data_each_config_latex[("BIC", "default")])

# Generate LaTeX table with proper formatting
latex_str = data_each_config_latex.to_latex()

print(latex_str)

/tmp/ipykernel_2776364/79301845.py:46: PerformanceWarning: indexing past lexsort depth may impact performance.
  display(data_each_config_latex[("BIC", "default")])


\mujs                          \#65                     \SLIDER{\mysliderlen}{1.00}{circle}
                               \#141                    \SLIDER{\mysliderlen}{0.67}{circle}
                               \#145                    \SLIDER{\mysliderlen}{0.87}{circle}
                               \#166                    \SLIDER{\mysliderlen}{0.37}{circle}
\libxml                        \#535                    \SLIDER{\mysliderlen}{1.00}{circle}
                               \#550                    \SLIDER{\mysliderlen}{0.40}{circle}
                               \#553                    \SLIDER{\mysliderlen}{1.00}{circle}
                               \#720                    \SLIDER{\mysliderlen}{0.33}{circle}
                               \#841                    \SLIDER{\mysliderlen}{1.00}{circle}
\poppler                       \#1282                   \SLIDER{\mysliderlen}{0.37}{circle}
                               \#1289                   \SLIDER{\mysliderlen}{0.

\begin{tabular}{llllllllllllllllllll}
\toprule
 & scenario & \multicolumn{9}{r}{BIC} & \multicolumn{9}{r}{FIX} \\
 & config_category & default & \multicolumn{3}{r}{prompt} & \multicolumn{3}{r}{llm} & \multicolumn{2}{r}{exec} & default & \multicolumn{3}{r}{prompt} & \multicolumn{3}{r}{llm} & \multicolumn{2}{r}{exec} \\
 & config &  & msg & diff & gen & temp1 & mini & deepseek & iter10 & xfeedback &  & msg & diff & gen & temp1 & mini & deepseek & iter10 & xfeedback \\
\midrule
\multirow[t]{4}{*}{\mujs} & \#65 & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{0.20}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{1.00}{circle} & \SLIDER{\mysliderlen}{0.90}{circle} & \SLIDER{\mysliderlen}{0.73}{circle} & \SLIDER{\mysliderlen}{0.33}{circle} & \SLIDER{\mysliderlen}{0.73}{circle} & \SLIDER{\mysliderlen}{0.73}{c

In [11]:
# NOTE: RQ2 analysis deepseek-r1 more success
# subset the data regarding the config
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]

data_def_success = data_def[data_def['success'] == 1]
data_def_success_unique = data_def_success.drop_duplicates(subset=['subject', 'iid', 'commit'])
data_def_success_unique = data_def_success_unique[['scenario', 'subject', 'iid', 'commit']]
display(data_def_success_unique)
data_def_success_unique_scenario = data_def_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_def_success_unique_scenario)

data_r1 = data[data['LLM'] == 'deepseek-r1']
# count success of msgonly
data_r1_success = data_r1[data_r1['success'] == 1]
# unique by commit
data_r1_success_unique = data_r1_success.drop_duplicates(subset=['subject', 'iid', 'commit'])
data_r1_success_unique = data_r1_success_unique[['scenario', 'subject', 'iid', 'commit']]
display(data_r1_success_unique)
# group by scenario
data_r1_success_unique_scenario = data_r1_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_r1_success_unique_scenario)
# group by scenario-subject
data_r1_success_unique_subject = data_r1_success_unique.groupby(['scenario', 'subject']).size().reset_index(name='count')
display(data_r1_success_unique_subject)

# diff specific iids where deepseek-r1 found bug but default did not, or vice versa
def_success_iids_bic = set(data_def_success_unique[data_def_success_unique['scenario'] == 'BIC']['iid'])
def_success_iids_fix = set(data_def_success_unique[data_def_success_unique['scenario'] == 'FIX']['iid'])
r1_success_iids_bic = set(data_r1_success_unique[data_r1_success_unique['scenario'] == 'BIC']['iid'])
r1_success_iids_fix = set(data_r1_success_unique[data_r1_success_unique['scenario'] == 'FIX']['iid'])

r1_not_def_bic = r1_success_iids_bic.difference(def_success_iids_bic)
def_not_r1_bic = def_success_iids_bic.difference(r1_success_iids_bic)
r1_not_def_fix = r1_success_iids_fix.difference(def_success_iids_fix)
def_not_r1_fix = def_success_iids_fix.difference(r1_success_iids_fix)

print(f"IIDs where deepseek-r1 found bug but default did not (BIC): {r1_not_def_bic}")
print(f"IIDs where default found bug but deepseek-r1 did not (BIC): {def_not_r1_bic}")
print(f"IIDs where deepseek-r1 found bug but default did not (FIX): {r1_not_def_fix}")
print(f"IIDs where default found bug but deepseek-r1 did not (FIX): {def_not_r1_fix}")


,scenario,subject,iid,commit
141,BIC,jerryscript,5138,70e275e
486,FIX,jerryscript,5117,d7e2125
842,BIC,jq,3262,0e067ef
1241,FIX,jq,2976,71c2ab5
1242,FIX,jq,3262,de21386
1670,BIC,libxml2,535,9a82b94
1672,BIC,libxml2,553,1406b20
1674,BIC,libxml2,841,9cfc723
2170,FIX,libxml2,535,d0c3f01
2172,FIX,libxml2,553,db21cd5


,scenario,count
0,BIC,7
1,FIX,13


,scenario,subject,iid,commit
440,FIX,jerryscript,5013,3686410
441,FIX,jerryscript,5138,95cc5e9
446,FIX,jerryscript,5117,d7e2125
802,BIC,jq,3262,0e067ef
815,BIC,jq,49014,4003202
1201,FIX,jq,2976,71c2ab5
1202,FIX,jq,3262,de21386
1204,FIX,jq,2825,3fa10e8
1223,FIX,jq,49014,499c91b
1620,BIC,libxml2,535,9a82b94


,scenario,count
0,BIC,10
1,FIX,20


,scenario,subject,count
0,BIC,mujs,2
1,BIC,libxml2,3
2,BIC,poppler,1
3,BIC,php-src,1
4,BIC,jq,2
5,BIC,micropython,1
6,FIX,mujs,3
7,FIX,libxml2,3
8,FIX,poppler,1
9,FIX,jerryscript,3


IIDs where deepseek-r1 found bug but default did not (BIC): {1289, 16978, 49014, 17847}
IIDs where default found bug but deepseek-r1 did not (BIC): {5138}
IIDs where deepseek-r1 found bug but default did not (FIX): {2825, 13007, 17841, 5138, 13041, 5013, 49014, 17847}
IIDs where default found bug but deepseek-r1 did not (FIX): {141}


Impact of iter 5 -> 10

In [12]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# set index [scenario, subject, iid]
data_def = data_def.set_index(["scenario", "subject", "iid"])
# only FIX scenario
data_def = data_def.loc["FIX"]

config_def10 = {
    "git_info": "FULL",
    "max_iter": 10,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def10 = data.copy()
for key, value in config_def10.items():
    if pd.isna(value):
        data_def10 = data_def10[pd.isna(data_def10[key])]
    else:
        data_def10 = data_def10[data_def10[key] == value]
# set index [scenario, subject, iid]
data_def10 = data_def10.set_index(["scenario", "subject", "iid"])
# only FIX scenario
data_def10 = data_def10.loc["FIX"]

# merge the two dataframes
data_def = pd.merge(
    data_def,
    data_def10,
    on=["subject", "iid"],
    suffixes=("_5", "_10"),
)
# check the case where the score is different
# data_def_diff = data_def[
#     data_def["score_5"] != data_def["score_10"]
# ]
display(data_def[['score_5', 'score_10']])
# check the average for each (scenario, subject, iid)
data_def_diff_avg = data_def.groupby(["subject", "iid"]).agg(
    {"score_5": "mean", "score_10": "mean"}
)
display(data_def_diff_avg)
# count the number of 0, 1, 2, 3 in the score_5 and score_10 for each 
# (scenario, subject, iid)
data_def_diff_count = data_def.groupby(["subject", "iid"]).agg(
    {"score_5": "value_counts", "score_10": "value_counts"}
)
data_def_diff_count = data_def_diff_count.unstack()
# data_def_diff_count.columns = data_def_diff_count.columns.swaplevel(0, 1)
data_def_diff_count = data_def_diff_count.sort_index(axis=1)
data_def_diff_count

score_5  score_10
subject     iid                    
jerryscript 5013        1         1
            5013        1         1
            5013        1         1
            5013        1         1
            5013        1         1
...                   ...       ...
z3          7252        0         0
            7252        0         1
            7252        0         0
            7252        0         1
            7252        0         0

[3600 rows x 2 columns]

score_5  score_10
subject     iid                     
mujs        65         2.2       2.2
            141        2.1       1.9
            145        2.8       3.0
            166        3.0       3.0
libxml2     535        3.0       3.0
            550        0.0       0.0
            553        3.0       3.0
            720        0.7       0.9
            841        3.0       3.0
poppler     1282       0.9       1.0
            1289       2.0       2.6
            1303       0.2       1.0
            1305       0.0       0.0
            1381       0.0       0.0
jerryscript 5013       1.0       1.0
            5117       2.0       1.6
            5138       1.0       1.4
            5153       0.0       0.0
z3          6659       0.0       0.0
            6914       0.0       0.1
            7246       0.0       0.0
            7252       0.5       0.8
php-src     16777      1.0       0.9
            16978      0.0       0.0
            17442      0.0       0.0
            17463      3.0       3.0
jq          2825       1.1       1.1
            2976       3.0       3.0
            3262       3.0       3.0
            49014      0.0       0.0
micropython 13007      0.2       0.3
            13041      2.0       2.0
            17733      1.0       1.0
            17815      3.0       3.0
            17841      1.0       1.0
            17847      0.0       0.0

score_10                      score_5                     
                         0      1      2      3       0      1      2      3
subject     iid                                                             
mujs        65         NaN   40.0    NaN   60.0     NaN   40.0    NaN   60.0
            141       20.0    NaN   50.0   30.0     NaN    NaN   90.0   10.0
            145        NaN    NaN    NaN  100.0     NaN   10.0    NaN   90.0
            166        NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
libxml2     535        NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
            550      100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
            553        NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
            720       10.0   90.0    NaN    NaN    30.0   70.0    NaN    NaN
            841        NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
poppler     1282       NaN  100.0    NaN    NaN    10.0   90.0    NaN    NaN
            1289       NaN   20.0    NaN   80.0    20.0   20.0    NaN   60.0
            1303      50.0    NaN   50.0    NaN    90.0    NaN   10.0    NaN
            1305     100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
            1381     100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
jerryscript 5013       NaN  100.0    NaN    NaN     NaN  100.0    NaN    NaN
            5117       NaN   70.0    NaN   30.0     NaN   50.0    NaN   50.0
            5138       NaN   80.0    NaN   20.0     NaN  100.0    NaN    NaN
            5153     100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
z3          6659     100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
            6914      90.0   10.0    NaN    NaN   100.0    NaN    NaN    NaN
            7246     100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
            7252      40.0   50.0    NaN   10.0    50.0   50.0    NaN    NaN
php-src     16777     10.0   90.0    NaN    NaN     NaN  100.0    NaN    NaN
            16978    100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
            17442    100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
            17463      NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
jq          2825       NaN   90.0   10.0    NaN     NaN   90.0   10.0    NaN
            2976       NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
            3262       NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
            49014    100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN
micropython 13007     90.0    NaN    NaN   10.0    90.0    NaN   10.0    NaN
            13041      NaN    NaN  100.0    NaN     NaN    NaN  100.0    NaN
            17733      NaN  100.0    NaN    NaN     NaN  100.0    NaN    NaN
            17815      NaN    NaN    NaN  100.0     NaN    NaN    NaN  100.0
            17841      NaN  100.0    NaN    NaN     NaN  100.0    NaN    NaN
            17847    100.0    NaN    NaN    NaN   100.0    NaN    NaN    NaN

### (New RQ3): Impact of ENHANCED/REDUCED msg

In [13]:
config_prompt_msg = config_def.copy()
config_prompt_msg["git_info"] = "MSGONLY"
config_prompt_diff = config_def.copy()
config_prompt_diff["git_info"] = "DIFFONLY"
config_prompt_enhanced = config_def.copy()
config_prompt_enhanced["git_info"] = "ENHANCED"
config_prompt_reduced = config_def.copy()
config_prompt_reduced["git_info"] = "REDUCED"


configs = {
    "default-": config_def,
    "prompt-msg": config_prompt_msg,
    "prompt-enhanced": config_prompt_enhanced,
    "prompt-reduced": config_prompt_reduced,
}

data_each_config = []
for config_key, config in configs.items():
    data_config = data.copy()
    for key, value in config.items():
        if pd.isna(value):
            data_config = data_config[pd.isna(data_config[key])]
        else:
            data_config = data_config[data_config[key] == value]
    data_config = data_config[data_config["scenario"] == "FIX"]
    # check the number of datapoints for each config
    print(f"{config_key}: {len(data_config)}")
    if len(data_config) != N_IID * 10:
        print(
            f"!!Warning!! {config_key} has {len(data_config)} data points; expected {N_IID * 10}"
        )
    data_config = (
        data_config.groupby(["scenario", "subject", "iid"])
        .agg({"score": "mean"})
        .reset_index()
    )
    data_config["config_category"] = config_key.split("-")[0]
    data_config["config"] = config_key.split("-")[1]
    data_each_config.append(data_config)

data_each_config = pd.concat(data_each_config)
data_each_config = data_each_config.pivot(
    index=["scenario", "subject", "iid"],
    columns=["config_category", "config"],
    values="score",
)

dfs = []
# add average row for each scenario
for scenario in data_each_config.index.levels[0]:
    print(f"Scenario: {scenario}")
    # scenario is the first level of the index
    sub_data = data_each_config.loc[
        data_each_config.index.get_level_values(0) == scenario
    ]
    average = sub_data.agg("mean").to_frame().T
    average.index = pd.MultiIndex.from_tuples([(scenario, "Average", "")])
    # data_each_config = pd.concat([data_each_config, average])
    dfs = dfs + [sub_data, average]
# data_each_config = data_each_config.sort_index()
data_each_config = pd.concat(dfs)
# reorder the index level 1: ["mujs", "libxml2", "poppler", "jerryscript", "php-src", "z3", "Average"]
data_each_config.index = pd.MultiIndex.from_arrays([
    pd.Categorical(data_each_config.index.get_level_values(0), categories=[
        "FIX"  # No BIC for enhanced/reduced
    ], ordered=True),
    pd.Categorical(data_each_config.index.get_level_values(1), categories=[
        "mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "jq", "micropython", "Average"
    ], ordered=True),
    data_each_config.index.get_level_values(2)
])
data_each_config = data_each_config.sort_index()
data_each_config

default-: 360
prompt-msg: 360
prompt-enhanced: 360
prompt-reduced: 360
Scenario: FIX


config_category         default    prompt                  
config                                msg  enhanced reduced
FIX mujs        65     2.200000  1.000000  3.000000   1.000
                141    2.100000  0.000000  3.000000   0.000
                145    2.800000  2.400000  3.000000   1.800
                166    3.000000  3.000000  3.000000   1.600
    libxml2     535    3.000000  3.000000  3.000000   3.000
                550    0.000000  0.000000  0.000000   0.000
                553    3.000000  3.000000  3.000000   3.000
                720    0.700000  1.000000  1.000000   0.900
                841    3.000000  3.000000  3.000000   3.000
    poppler     1282   0.900000  1.000000  0.900000   1.000
                1289   2.000000  0.000000  2.100000   0.000
                1303   0.200000  0.000000  1.600000   0.000
                1305   0.000000  0.000000  0.000000   0.100
                1381   0.000000  0.000000  0.000000   0.000
    jerryscript 5013   1.000000  1.000000  1.000000   0.000
                5117   2.000000  1.400000  2.800000   1.000
                5138   1.000000  1.000000  3.000000   1.000
                5153   0.000000  0.000000  1.200000   0.000
    z3          6659   0.000000  0.000000  0.900000   0.000
                6914   0.000000  0.200000  0.700000   0.000
                7246   0.000000  0.000000  0.000000   0.000
                7252   0.500000  0.000000  1.600000   0.000
    php-src     16777  1.000000  0.600000  0.700000   0.000
                16978  0.000000  0.000000  0.200000   0.000
                17442  0.000000  0.000000  0.000000   0.000
                17463  3.000000  3.000000  3.000000   3.000
    jq          2825   1.100000  1.000000  1.200000   1.000
                2976   3.000000  3.000000  3.000000   1.000
                3262   3.000000  2.800000  2.800000   3.000
                49014  0.000000  0.000000  0.000000   0.000
    micropython 13007  0.200000  0.500000  1.000000   0.000
                13041  2.000000  2.100000  2.000000   2.000
                17733  1.000000  1.000000  1.000000   1.000
                17815  3.000000  3.000000  3.000000   0.300
                17841  1.000000  1.000000  1.200000   1.000
                17847  0.000000  0.000000  2.400000   0.000
    Average            1.269444  1.083333  1.647222   0.825

In [14]:
# to latex
data_each_config_latex = data_each_config.copy()
# index formats
data_each_config_latex.index = data_each_config_latex.index.set_levels(
    [
        data_each_config_latex.index.levels[0],
        data_each_config_latex.index.levels[1].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "jq": r"\jq",
                "micropython": r"\micropython",
                "Average": r"\multicolumn{2}{c|}{Average}",
            }
        ),
        data_each_config_latex.index.levels[2].map(
            lambda x: f"\\#{x}" if x else ""
        ),
    ]
)
###### column formats
# score -> \SLIDER{3.3em}{score / 3:.2f}{circle}
data_each_config_latex = data_each_config_latex.map(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
display(data_each_config_latex)
print(data_each_config_latex.to_latex().replace("[t]", "").replace(
    r"\multirow{24}{*}{BIC}", r"\multirow{24}{*}{\begin{sideways}Bug-finding\end{sideways}}"
).replace(
    r"\multirow{24}{*}{FIX}", r"\multirow{24}{*}{\begin{sideways}Bug-reproduction\end{sideways}}"
).replace(
    r"\multicolumn{2}{c|}{Average} &  &",
    r"\multicolumn{2}{c|}{Average} &"
).replace(
    r"\cline{1-11} \cline{2-11}", r"\midrule"
).replace(
    r"\cline{2-11}", r"\cmidrule{2-11}"
))

config_category                                                default  \
config                                                                   
FIX \mujs                        \#65     \SLIDER{3.3em}{0.73}{circle}   
                                 \#141    \SLIDER{3.3em}{0.70}{circle}   
                                 \#145    \SLIDER{3.3em}{0.93}{circle}   
                                 \#166    \SLIDER{3.3em}{1.00}{circle}   
    \libxml                      \#535    \SLIDER{3.3em}{1.00}{circle}   
                                 \#550    \SLIDER{3.3em}{0.00}{circle}   
                                 \#553    \SLIDER{3.3em}{1.00}{circle}   
                                 \#720    \SLIDER{3.3em}{0.23}{circle}   
                                 \#841    \SLIDER{3.3em}{1.00}{circle}   
    \poppler                     \#1282   \SLIDER{3.3em}{0.30}{circle}   
                                 \#1289   \SLIDER{3.3em}{0.67}{circle}   
                                 \#1303   \SLIDER{3.3em}{0.07}{circle}   
                                 \#1305   \SLIDER{3.3em}{0.00}{circle}   
                                 \#1381   \SLIDER{3.3em}{0.00}{circle}   
    \jerryscript                 \#5013   \SLIDER{3.3em}{0.33}{circle}   
                                 \#5117   \SLIDER{3.3em}{0.67}{circle}   
                                 \#5138   \SLIDER{3.3em}{0.33}{circle}   
                                 \#5153   \SLIDER{3.3em}{0.00}{circle}   
    \zthree                      \#6659   \SLIDER{3.3em}{0.00}{circle}   
                                 \#6914   \SLIDER{3.3em}{0.00}{circle}   
                                 \#7246   \SLIDER{3.3em}{0.00}{circle}   
                                 \#7252   \SLIDER{3.3em}{0.17}{circle}   
    \php                         \#16777  \SLIDER{3.3em}{0.33}{circle}   
                                 \#16978  \SLIDER{3.3em}{0.00}{circle}   
                                 \#17442  \SLIDER{3.3em}{0.00}{circle}   
                                 \#17463  \SLIDER{3.3em}{1.00}{circle}   
    \jq                          \#2825   \SLIDER{3.3em}{0.37}{circle}   
                                 \#2976   \SLIDER{3.3em}{1.00}{circle}   
                                 \#3262   \SLIDER{3.3em}{1.00}{circle}   
                                 \#49014  \SLIDER{3.3em}{0.00}{circle}   
    \micropython                 \#13007  \SLIDER{3.3em}{0.07}{circle}   
                                 \#13041  \SLIDER{3.3em}{0.67}{circle}   
                                 \#17733  \SLIDER{3.3em}{0.33}{circle}   
                                 \#17815  \SLIDER{3.3em}{1.00}{circle}   
                                 \#17841  \SLIDER{3.3em}{0.33}{circle}   
                                 \#17847  \SLIDER{3.3em}{0.00}{circle}   
    \multicolumn{2}{c|}{Average}          \SLIDER{3.3em}{0.42}{circle}   

config_category                                                 prompt  \
config                                                             msg   
FIX \mujs                        \#65     \SLIDER{3.3em}{0.33}{circle}   
                                 \#141    \SLIDER{3.3em}{0.00}{circle}   
                                 \#145    \SLIDER{3.3em}{0.80}{circle}   
                                 \#166    \SLIDER{3.3em}{1.00}{circle}   
    \libxml                      \#535    \SLIDER{3.3em}{1.00}{circle}   
                                 \#550    \SLIDER{3.3em}{0.00}{circle}   
                                 \#553    \SLIDER{3.3em}{1.00}{circle}   
                                 \#720    \SLIDER{3.3em}{0.33}{circle}   
                                 \#841    \SLIDER{3.3em}{1.00}{circle}   
    \poppler                     \#1282   \SLIDER{3.3em}{0.33}{circle}   
                                 \#1289   \SLIDER{3.3em}{0.00}{circle}   
                                 \#1303   \SLIDER{3.3em}{0.00}{circle}   
                                 \#1305   \SLIDER{3.3em}{0.00}{circle}   
   

\begin{tabular}{lllllll}
\toprule
 &  & config_category & default & \multicolumn{3}{r}{prompt} \\
 &  & config &  & msg & enhanced & reduced \\
\midrule
\multirow{37}{*}{FIX} & \multirow{4}{*}{\mujs} & \#65 & \SLIDER{3.3em}{0.73}{circle} & \SLIDER{3.3em}{0.33}{circle} & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{0.33}{circle} \\
 &  & \#141 & \SLIDER{3.3em}{0.70}{circle} & \SLIDER{3.3em}{0.00}{circle} & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{0.00}{circle} \\
 &  & \#145 & \SLIDER{3.3em}{0.93}{circle} & \SLIDER{3.3em}{0.80}{circle} & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{0.60}{circle} \\
 &  & \#166 & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{0.53}{circle} \\
\cline{2-7}
 & \multirow{5}{*}{\libxml} & \#535 & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{1.00}{circle} & \SLIDER{3.3em}{1.00}{circle} \\
 &  & \#550 & \SLIDER{3.3em}{0.00}{circle} & \SLIDER{3.3em}{0.00}{circle} 

# Table 4

## WAFLGo results

In [15]:
import glob

# init_result = [
#     ["BIC", "mujs", 65, "F"],
#     ["BIC", "mujs", 141, "F"],
#     ["BIC", "mujs", 145, "F"],
#     ["BIC", "mujs", 166, "B"],
#     ["BIC", "libxml2", 535, "B"],
#     ["BIC", "libxml2", 550, "R"],
#     ["BIC", "poppler", 1282, "R"],
#     ["BIC", "poppler", 1289, "R"],
#     ["BIC", "poppler", 1303, "F"],
#     ["BIC", "poppler", 1305, "F"],
#     ["BIC", "poppler", 1381, "R"],
#     ["FIX", "mujs", 65, "R"],
#     ["FIX", "mujs", 141, "F"],
#     ["FIX", "mujs", 145, "F"],
#     ["FIX", "mujs", 166, "B"],
#     ["FIX", "libxml2", 535, "B"],
#     ["FIX", "libxml2", 550, "F"],
#     ["FIX", "poppler", 1282, "R"],
#     ["FIX", "poppler", 1289, "R"],
#     ["FIX", "poppler", 1303, "F"],
#     ["FIX", "poppler", 1305, "R"],
#     ["FIX", "poppler", 1381, "R"],
# ]


def parse_init(data_path):
    data = pd.read_csv(data_path, sep=r"\s\|\s", engine="python")
    data["scenario"] = data_path.split("_")[-2]
    data["subject"] = data_path.split("_")[-3]
    data["init"] = data["final"].map(
        {"N": "F", "X": "F", "B": "B", "D": "D", "R": "R"}
    )
    data["iid"] = data["issue"]
    return data[["scenario", "subject", "iid", "init"]]


# parse_init("/home/nfs0/smbshares/softsec/seedscheck_mujs_BIC_withiid.csv")
init_result = pd.concat(
    [parse_init(f) for f in glob.glob("fuzz/seedscheck_*_withiid.csv")]
)
init_result.set_index(["scenario", "subject", "iid"], inplace=True)
init_result.sort_index(inplace=True)
init_result

init
scenario subject     iid       
BIC      jerryscript 5013     F
                     5117     R
                     5138     R
                     5153     R
         jq          2825     R
                     2976     R
                     3262     B
                     49014    F
         libxml2     535      B
                     550      D
                     553      B
                     720      D
                     841      B
         micropython 13007    R
                     13041    F
                     17733    R
                     17815    R
                     17841    F
                     17847    F
         mujs        65       F
                     141      F
                     145      F
                     166      B
         poppler     1282     R
                     1289     R
                     1303     F
                     1305     F
                     1381     R
FIX      jerryscript 5013     F
                     5117     R
                     5138     F
                     5153     F
         jq          2825     R
                     2976     R
                     3262     B
                     49014    F
         libxml2     535      B
                     550      F
                     553      B
                     720      D
                     841      B
         micropython 13007    D
                     13041    D
                     17733    D
                     17815    D
                     17841    D
                     17847    D
         mujs        65       R
                     141      F
                     145      F
                     166      B
         poppler     1282     R
                     1289     R
                     1303     F
                     1305     R
                     1381     R

In [16]:
# parse WAFLGo result

# index = ["scenario", "iid"]
# columns = [["WAFLGo", "WAFLGo"], ["Result", "Time"]]

# Function to parse each waflgo_*.csv file
import glob


def parse_fuzz_csv(file_path):
    try:
        df = pd.read_csv(file_path, sep=r"\t", engine="python")
    except Exception as e:  # use different separator
        df = pd.read_csv(file_path, sep=r"\s\|\s", engine="python")
    print(f"Reading {file_path} with shape {df.shape}")
    df.columns = df.columns.str.strip()
    if 'scenario' not in df.columns:
        df["scenario"] = file_path.split(".")[0].split("_")[
            -2
        ]  # Extract scenario from file name
    if 'subject' not in df.columns:
        df["subject"] = file_path.split(".")[0].split("_")[
            1
        ]  # Extract subject from file name
    df["iid"] = df["issue"]
    df["success"] = df["final"].apply(
        lambda x: 1 if x == "B" else 0
    )  # Success if final is 'B'
    try:
        df["time"] = (
            df["time to find first crash"].apply(pd.to_numeric, errors="coerce")
            / 1000
        )  # Convert time to seconds
    except KeyError:  # postfuzz_*.csv
        df["time"] = (
            df["time to find first crash in ms"].apply(pd.to_numeric, errors="coerce")
            / 1000
        )  # Convert time to seconds
    df.loc[df["success"] == 0, "time"] = (
        np.nan
    )  # some rows have time but not success, should be NaN
    # only keep columns: scenario, subject, iid, commit, idx, success, time
    df = df[["scenario", "subject", "iid", "commit", "idx", "success", "time"]]
    return df


# Parse and concatenate all CSV files
# csv_files = glob.glob("fuzz/waflgo_*_withiid.csv")
# waflgo_data = pd.concat([parse_fuzz_csv(file) for file in csv_files])
waflgo_data = parse_fuzz_csv("fuzz/merged_waflgo.csv")
display(waflgo_data.head())
# Group by scenario and iid, and aggregate success and time
waflgo_result = (
    waflgo_data.groupby(["scenario", "subject", "iid"])
    .agg(
        {
            "success": "sum",  # Number of successes
            "time": "mean",  # Average time in seconds
        }
    )
    .reset_index()
)

# Set index as scenario and iid
waflgo_result.set_index(["scenario", "subject", "iid"], inplace=True)

# Merge with initial results
waflgo_result = waflgo_result.join(init_result, how="outer")
# Reorder columns
waflgo_result = waflgo_result[["init", "success", "time"]]
# for the cases there init is B, set success 10, also time to T.O. as some case like jq 3262 somehow still launch WAFLGo, though seed should already crash
waflgo_result.loc[waflgo_result["init"] == "B", "success"] = 10
waflgo_result.loc[waflgo_result["init"] == "B", "time"] = 24 * 3600
# for the cases there success is not NaN but time is NaN, set time 24 hours
waflgo_result.loc[
    waflgo_result["success"].notna() & waflgo_result["time"].isna(), "time"
] = 24 * 3600
# rename columns
waflgo_result.columns = pd.MultiIndex.from_product([["WAFLGo"], ["Init", "Bug", "Time"]])
waflgo_result

Reading fuzz/merged_waflgo.csv with shape (490, 9)


,scenario,subject,iid,commit,idx,success,time
0,BIC,mujs,65,8c27b12,1,1,155.094
1,BIC,mujs,65,8c27b12,2,1,20353.874
2,BIC,mujs,65,8c27b12,3,1,105.214
3,BIC,mujs,65,8c27b12,4,1,7825.728
4,BIC,mujs,65,8c27b12,5,1,463.886


WAFLGo                  
                             Init   Bug        Time
scenario subject     iid                           
BIC      jerryscript 5013       F   NaN         NaN
                     5117       R   4.0    782.7990
                     5138       R   0.0  86400.0000
                     5153       R   0.0  86400.0000
         jq          2825       R   0.0  86400.0000
                     2976       R   2.0   6903.9730
                     3262       B  10.0  86400.0000
                     49014      F   0.0  86400.0000
         libxml2     535        B  10.0  86400.0000
                     550        D   0.0  86400.0000
                     553        B  10.0  86400.0000
                     720        D   0.0  86400.0000
                     841        B  10.0  86400.0000
         micropython 13007      R   0.0  86400.0000
                     13041      F   0.0  86400.0000
                     17733      R   0.0  86400.0000
                     17815      R   0.0  86400.0000
                     17841      F   NaN         NaN
                     17847      F   0.0  86400.0000
         mujs        65         F  10.0  17036.2719
                     141        F   0.0  86400.0000
                     145        F  10.0   1041.3056
                     166        B  10.0  86400.0000
         poppler     1282       R   6.0    383.0208
                     1289       R   0.0  86400.0000
                     1303       F   0.0  86400.0000
                     1305       F   0.0  86400.0000
                     1381       R   0.0  86400.0000
FIX      jerryscript 5013       F   0.0  86400.0000
                     5117       R   0.0  86400.0000
                     5138       F   0.0  86400.0000
                     5153       F   NaN         NaN
         jq          2825       R   0.0  86400.0000
                     2976       R   1.0  68618.2100
                     3262       B  10.0  86400.0000
                     49014      F   0.0  86400.0000
         libxml2     535        B  10.0  86400.0000
                     550        F   0.0  86400.0000
                     553        B  10.0  86400.0000
                     720        D   0.0  86400.0000
                     841        B  10.0  86400.0000
         micropython 13007      D   NaN         NaN
                     13041      D   0.0  86400.0000
                     17733      D   0.0  86400.0000
                     17815      D   NaN         NaN
                     17841      D   NaN         NaN
                     17847      D   0.0  86400.0000
         mujs        65         R   0.0  86400.0000
                     141        F   5.0    767.1130
                     145        F  10.0    506.1808
                     166        B  10.0  86400.0000
         poppler     1282       R   0.0  86400.0000
                     1289       R   0.0  86400.0000
                     1303       F   0.0  86400.0000
                     1305       R   0.0  86400.0000
                     1381       R   0.0  86400.0000

## Cleverest results

In [17]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# check the number of datapoints; expected to = # of iid * 2 scenario * 10 repetition; if not, print warning
if len(data_def) != N_DATAPOINT:
    print(
        f"!!Warning!!: the number of datapoints is {len(data_def)}; expected to be {N_DATAPOINT}"
    )
# aggregate the data regarding the scenario and iid
data_def["reached"] = data_def["score"].apply(lambda x: 1 if x >= 1 else 0)
data_def["changed"] = data_def["score"].apply(lambda x: 1 if x >= 2 else 0)

data_def = (
    data_def.groupby(["scenario", "subject", "iid"])
    # .agg({"score": "mean", "success": "sum", "time": "mean"})
    .agg({"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"})
    .reset_index()
)

display(data_def.tail())
data_def_tb4 = data_def.copy()
data_def_tb4 = data_def_tb4.groupby(["scenario", "subject", "iid"]).agg(
    # {"score": "mean", "success": "sum", "time": "mean"}
    {"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"}
)
# rename columns
data_def_tb4.columns = pd.MultiIndex.from_product([["Cleverest"], ["Score", "Reach", "Change", "Bug", "Time"]])
data_def_tb4

,scenario,subject,iid,score,reached,changed,success,time
67,FIX,micropython,13041,2.0,10,10,0,21.8
68,FIX,micropython,17733,1.0,10,0,0,31.4
69,FIX,micropython,17815,3.0,10,10,10,5.5
70,FIX,micropython,17841,1.0,10,0,0,39.2
71,FIX,micropython,17847,0.0,0,0,0,29.2


Cleverest                       
                               Score Reach Change Bug  Time
scenario subject     iid                                   
BIC      mujs        65          3.0    10     10  10   7.5
                     141         2.0    10     10   0  45.4
                     145         2.6    10      8   8  37.1
                     166         1.1    10      1   0  57.6
         libxml2     535         3.0    10     10  10   5.4
...                              ...   ...    ...  ..   ...
FIX      micropython 13041       2.0    10     10   0  21.8
                     17733       1.0    10      0   0  31.4
                     17815       3.0    10     10  10   5.5
                     17841       1.0    10      0   0  39.2
                     17847       0.0     0      0   0  29.2

[72 rows x 5 columns]

## ClevFuzz results

In [18]:
data_def[(data_def["subject"] == "poppler") & (data_def["scenario"] == "BIC") & (data_def["iid"] == 1282)]

,scenario,subject,iid,score,reached,changed,success,time
9,BIC,poppler,1282,1.1,8,3,0,196.6


In [19]:
data_def = data.copy()
data_def

,file,scenario,git_info,max_iter,LLM,LLM_temp,NOFEEDBACK,GENCMD,RUNFUZZ,subject,commit,final_result,unintended_bug,time,iid,score,success
0,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,100,5013,1,0
1,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,D,False,97,5138,2,0
2,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,53b61c1,D,False,57,5117,2,0
3,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,B,False,36,5153,3,1
4,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,107,5013,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6755,SUMMARY_z3_FIX_GITREDUCED_ITER5_gpt-4o-2024-08...,FIX,REDUCED,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,z3,a6b5027,N,False,546,7252,0,0
6756,SUMMARY_z3_FIX_GITREDUCED_ITER5_gpt-4o-2024-08...,FIX,REDUCED,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,z3,a849a29,N,False,62,6659,0,0
6757,SUMMARY_z3_FIX_GITREDUCED_ITER5_gpt-4o-2024-08...,FIX,REDUCED,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,z3,eea9c0b,N,False,126,6914,0,0
6758,SUMMARY_z3_FIX_GITREDUCED_ITER5_gpt-4o-2024-08...,FIX,REDUCED,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,z3,49610f5,N,False,132,7246,0,0


In [20]:
# Read all postfuzz_*.csv files
# csv_files = glob.glob("fuzz/postfuzz_*_withiid.csv")
# print(csv_files)
# csv_files = [f for f in csv_files if "withiid" not in f]
# fuzz_data = pd.concat([parse_fuzz_csv(file) for file in csv_files])
fuzz_data = parse_fuzz_csv("fuzz/merged_postfuzz.csv")
display(fuzz_data.head())


# for each row, find row from data_def with the 'file' includes the 'idx' in
# fuzz_data and 'iid' and 'commit' are the same; if not found, print error
clevfuzz_data = []
for idx, row in fuzz_data.iterrows():
    corr_row = data_def[
        (data_def["file"].str.contains(row["idx"]))
        & (data_def["iid"] == row["iid"])
        & (data_def["commit"] == row["commit"])
    ]
    if len(corr_row) != 1:
        print(f"Error: {len(corr_row)} rows found for {row['idx']}")
        # display full rows
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", None)
        pd.set_option("display.max_colwidth", None)
        display(corr_row)
        break
    # add to clevfuzz_data only if cleverest result >= "R", note some "X" also counts, so check score is safer
    try:
        if (corr_row["score"].values[0] >= 1 and corr_row["score"].values[0] < 3):
            # (
                # corr_row[["scenario", "subject"]].values[0] == ("BIC", "mujs")
            # ).all()
            # and (corr_row["iid"].values[0] == 141):
            new_row = row.copy()
            new_row["cleverest"] = corr_row["final_result"].values[0]
            clevfuzz_data.append(new_row)
    except KeyError:
        display(corr_row)
        break
pd.reset_option("display.max_columns")
pd.reset_option("display.width")
pd.reset_option("display.max_colwidth")
clevfuzz_data = pd.DataFrame(clevfuzz_data)
display(clevfuzz_data.head())
clevfuzz_data.set_index(["scenario", "subject", "iid"], inplace=True)
clevfuzz_data.sort_index(inplace=True)
# display(clevfuzz_data.loc[("BIC", "mujs", 141), :])
# display(clevfuzz_data.loc[("BIC", "poppler", 1282), :])
# display(clevfuzz_data.loc[("BIC", "mujs", 166), :])
# display(clevfuzz_data.loc[("FIX", "mujs", 65), :])

# set 24 hours to the cases where time is NaN
clevfuzz_data.loc[clevfuzz_data["time"].isna(), "time"] = 24 * 60 * 60
# add row 'try' to count the number of tries
clevfuzz_data["try"] = 1
clevfuzz_data

Reading fuzz/merged_postfuzz.csv with shape (481, 9)


,scenario,subject,iid,commit,idx,success,time
0,BIC,mujs,65,8c27b12,250905-2232,1,0.0
1,BIC,mujs,65,8c27b12,250905-2235,1,0.0
2,BIC,mujs,65,8c27b12,250905-2238,1,0.0
3,BIC,mujs,65,8c27b12,250905-2240,1,0.0
4,BIC,mujs,65,8c27b12,250905-2242,1,0.0


,scenario,subject,iid,commit,idx,success,time,cleverest
10,BIC,mujs,141,832e069,250905-2232,1,23863.771,X
11,BIC,mujs,141,832e069,250905-2235,1,4192.597,X
12,BIC,mujs,141,832e069,250905-2238,1,13406.357,X
13,BIC,mujs,141,832e069,250905-2240,1,37298.857,X
14,BIC,mujs,141,832e069,250905-2242,1,3357.036,X


commit          idx  success       time cleverest  \
scenario subject     iid                                                        
BIC      jerryscript 5013  b7e3bae  250826-2133        0  86400.000         R   
                     5013  b7e3bae  250826-2139        0  86400.000         R   
                     5013  b7e3bae  250826-2145        0  86400.000         X   
                     5013  b7e3bae  250826-2150        1    101.758         R   
                     5013  b7e3bae  250826-2156        0  86400.000         R   
...                            ...          ...      ...        ...       ...   
FIX      z3          7252  a6b5027  250826-2135        0  86400.000         R   
                     7252  a6b5027  250826-2140        0  86400.000         R   
                     7252  a6b5027  250826-2150        0  86400.000         R   
                     7252  a6b5027  250903-2341        0  86400.000         R   
                     7252  a6b5027  250903-2351        0  86400.000         R   

                           try  
scenario subject     iid        
BIC      jerryscript 5013    1  
                     5013    1  
                     5013    1  
                     5013    1  
                     5013    1  
...                        ...  
FIX      z3          7252    1  
                     7252    1  
                     7252    1  
                     7252    1  
                     7252    1  

[328 rows x 6 columns]

In [21]:
clevfuzz_result = clevfuzz_data.groupby(["scenario", "subject", "iid"]).agg(
    {"try": "sum", "success": "sum", "time": "mean"}
)

# bug: "{success}/{try}"
clevfuzz_result["bug"] = (
    clevfuzz_result["success"].astype(str)
    + "/"
    + clevfuzz_result["try"].astype(str)
)
clevfuzz_result = clevfuzz_result[["bug", "time", "success"]]
# rename columns
clevfuzz_result.columns = pd.MultiIndex.from_product(
    [["ClevFuzz"], ["Bug", "Time", "Success"]]
)
clevfuzz_result

ClevFuzz                      
                                Bug          Time Success
scenario subject     iid                                 
BIC      jerryscript 5013      1/10  77770.175800       1
                     5117     10/10    101.485500      10
                     5138       8/8     37.217000       8
                     5153     10/10     59.649400      10
         jq          2825     10/10  16954.793200      10
                     2976     10/10   4899.672700      10
                     49014      2/2  28284.083000       2
         libxml2     550       0/10  86400.000000       0
                     720       4/10  65281.121000       4
         micropython 13007      7/9  25632.405667       7
                     17733     6/10  45308.074000       6
                     17815     4/10  64474.658100       4
                     17841     0/10  86400.000000       0
                     17847     2/10  74021.192800       2
         mujs        141      10/10  12305.941600      10
                     145        2/2      0.490500       2
                     166       0/10  86400.000000       0
         php-src     16777     0/10  86400.000000       0
                     16978     5/10  48348.041600       5
                     17442     0/10  86400.000000       0
                     17463      0/6  86400.000000       0
         poppler     1282       8/8     10.569000       8
                     1289       3/4  21604.431750       3
         z3          6914       0/2  86400.000000       0
                     7246      0/10  86400.000000       0
                     7252       0/8  86400.000000       0
FIX      jerryscript 5013     10/10  10962.102700      10
                     5117       2/5  75283.696600       2
                     5138      4/10  64221.021600       4
         jq          2825       2/5  69631.101800       2
         libxml2     720        7/7  11351.540429       7
         micropython 13007      0/1  86400.000000       0
                     13041    10/10  10713.724300      10
                     17733     5/10  54018.080800       5
                     17841     0/10  86400.000000       0
         mujs        65         1/4  65022.534750       1
                     141        9/9   6212.566889       9
                     145        1/1      3.639000       1
         php-src     16777     0/10  86400.000000       0
         poppler     1282       1/9  79770.719222       1
                     1289       2/2      8.386000       2
                     1303       0/1  86400.000000       0
         z3          7252       0/5  86400.000000       0

In [22]:
# merge waflgo_result, data_def_tb4, and clevfuzz_result
tb4 = waflgo_result.join(data_def_tb4, how="outer")
tb4 = tb4.join(clevfuzz_result, how="outer")
tb4.loc[:, ("ClevFuzz", "BugAll")] = tb4.apply(
    lambda x: (
        x["Cleverest"]["Bug"]
        if pd.isna(x["ClevFuzz"]["Bug"])
        else x["ClevFuzz"]["Success"] + x["Cleverest"]["Bug"]
    ),
    axis=1,
)
# drop ["ClevFuzz", "Success"]
tb4 = tb4.drop(("ClevFuzz", "Success"), axis=1)

# sort the index: ["Scenario", "Subject", "Issue"]
tb4.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4.index.get_level_values(1),
            categories=[
                *ALL_SUBJECTS,
                "Aggregate",
            ],
            ordered=True,
        ),
        tb4.index.get_level_values(2),
    ],
    names=["Scenario", "Subject", "Issue"],
)
tb4 = tb4.sort_index()
tb4 = tb4[tb4["Cleverest"]["Score"].notna()]
# NOTE: keep only rows supported by WAFLGo, comment to get full table in supplementary
tb4 = tb4[tb4["WAFLGo"]["Time"].notna()]
tb4

WAFLGo                   Cleverest               \
                             Init   Bug        Time     Score Reach Change   
Scenario Subject     Issue                                                   
BIC      mujs        65         F  10.0  17036.2719       3.0    10     10   
                     141        F   0.0  86400.0000       2.0    10     10   
                     145        F  10.0   1041.3056       2.6    10      8   
                     166        B  10.0  86400.0000       1.1    10      1   
         libxml2     535        B  10.0  86400.0000       3.0    10     10   
                     550        D   0.0  86400.0000       1.2    10      2   
                     553        B  10.0  86400.0000       3.0    10     10   
                     720        D   0.0  86400.0000       1.0    10      0   
                     841        B  10.0  86400.0000       3.0    10     10   
         poppler     1282       R   6.0    383.0208       1.1     8      3   
                     1289       R   0.0  86400.0000       0.4     4      0   
                     1303       F   0.0  86400.0000       0.0     0      0   
                     1305       F   0.0  86400.0000       0.0     0      0   
                     1381       R   0.0  86400.0000       0.0     0      0   
         jerryscript 5117       R   4.0    782.7990       2.0    10     10   
                     5138       R   0.0  86400.0000       2.2    10     10   
                     5153       R   0.0  86400.0000       2.0    10     10   
         jq          2825       R   0.0  86400.0000       1.0    10      0   
                     2976       R   2.0   6903.9730       1.0    10      0   
                     3262       B  10.0  86400.0000       3.0    10     10   
                     49014      F   0.0  86400.0000       0.4     2      2   
         micropython 13007      R   0.0  86400.0000       1.6     9      7   
                     13041      F   0.0  86400.0000       0.0     0      0   
                     17733      R   0.0  86400.0000       1.0    10      0   
                     17815      R   0.0  86400.0000       1.9    10      9   
                     17847      F   0.0  86400.0000       2.0    10     10   
FIX      mujs        65         R   0.0  86400.0000       2.2    10      6   
                     141        F   5.0    767.1130       2.1    10     10   
                     145        F  10.0    506.1808       2.8    10      9   
                     166        B  10.0  86400.0000       3.0    10     10   
         libxml2     535        B  10.0  86400.0000       3.0    10     10   
                     550        F   0.0  86400.0000       0.0     0      0   
                     553        B  10.0  86400.0000       3.0    10     10   
                     720        D   0.0  86400.0000       0.7     7      0   
                     841        B  10.0  86400.0000       3.0    10     10   
         poppler     1282       R   0.0  86400.0000       0.9     9      0   
                     1289       R   0.0  86400.0000       2.0     8      6   
                     1303       F   0.0  86400.0000       0.2     1      1   
                     1305       R   0.0  86400.0000       0.0     0      0   
                     1381       R   0.0  86400.0000       0.0     0      0   
         jerryscript 5013       F   0.0  86400.0000       1.0    10      0   
                     5117       R   0.0  86400.0000       2.0    10      5   
                     5138       F   0.0  86400.0000       1.0    10      0   
         jq          2825       R   0.0  86400.0000       1.1    10      1   
                     2976       R   1.0  68618.2100       3.0    10     10   
                     3262       B  10.0  86400.0000       3.0    10     10   
                     49014      F   0.0  86400.0000       0.0     0      0   
         micropython 13041      D   0.0  86400.0000       2.0    10     10   
                     17733      D   0.0  86400.00

In [23]:
# NOTE: RQ4 main text numbers, can also manually count tb4

# For BIC only
tb4_bic = tb4.loc['BIC']
# For FIX only
tb4_fix = tb4.loc['FIX']
print("ClevFuzz BIC total commits:", tb4_bic["ClevFuzz"]["Time"].notna().sum())
print("ClevFuzz BIC total tries:", tb4_bic["ClevFuzz"]["Bug"].dropna().str.split("/").str[1].astype(int).sum())
print("ClevFuzz BIC total successes:", tb4_bic["ClevFuzz"]["Bug"].dropna().str.split("/").str[0].astype(int).sum())
print("ClevFuzz FIX total commits:", tb4_fix["ClevFuzz"]["Time"].notna().sum())
print("ClevFuzz FIX total tries:", tb4_fix["ClevFuzz"]["Bug"].dropna().str.split("/").str[1].astype(int).sum())
print("ClevFuzz FIX total successes:", tb4_fix["ClevFuzz"]["Bug"].dropna().str.split("/").str[0].astype(int).sum())

ClevFuzz BIC total commits: 17
ClevFuzz BIC total tries: 143
ClevFuzz BIC total successes: 96
ClevFuzz FIX total commits: 13
ClevFuzz FIX total tries: 83
ClevFuzz FIX total successes: 54


In [24]:
# create the aggregation row
agg_rows = []
for scenario in tb4.index.levels[0]:
    print(f"Scenario: {scenario}")
    sub_data = tb4.loc[tb4.index.get_level_values(0) == scenario]
    len_sub_data = len(sub_data)
    # waflgo_bug aggregation: count the number of non-NaN or non-zero values
    waflgo_bug = (sub_data["WAFLGo"]["Bug"] > 0).sum()
    # waflgo_time aggregation: mean of non-NaN values
    col = sub_data["WAFLGo"]["Time"]
    col_nonnan = col[~col.isna()]
    waflgo_time = col_nonnan.mean()
    # cleverest_score aggregation: mean of values
    cleverest_score = sub_data["Cleverest"]["Score"].mean()
    # cleverest_bug aggregation: count the number of non-zero values
    cleverest_bug = (sub_data["Cleverest"]["Bug"] > 0).sum()
    cleverest_change = (sub_data["Cleverest"]["Change"] > 0).sum()
    cleverest_reach = (sub_data["Cleverest"]["Reach"] > 0).sum()
    # cleverest_time aggregation: mean of values
    cleverest_time = sub_data["Cleverest"]["Time"].mean()
    # clevfuzz_bugall aggregation: count the number of non-zero values
    clevfuzz_bugall = (sub_data["ClevFuzz"]["BugAll"] > 0).sum()
    agg_rows.append(
        {
            # ("Scenario", "Subject", "Issue"): (scenario, "Aggregate", ""),
            "Scenario": scenario,
            "Subject": "Aggregate",
            "Issue": "",
            ("WAFLGo", "Init", ""): "",
            ("WAFLGo", "Bug", ""): f"{waflgo_bug}/{len_sub_data}",
            ("WAFLGo", "Time", ""): str(
                pd.to_datetime(waflgo_time, unit="s").strftime("%H:%M:%S")
            ),
            (
                "Cleverest",
                "Score",
                "",
            ): f"\\SLIDER{{3.3em}}{{{(cleverest_score/3):.2f}}}{{circle}}",
            ("Cleverest", "Reach", ""): f"{cleverest_reach}/{len_sub_data}",
            ("Cleverest", "Change", ""): f"{cleverest_change}/{len_sub_data}",
            ("Cleverest", "Bug", ""): f"{cleverest_bug}/{len_sub_data}",
            ("Cleverest", "Time", ""): str(
                pd.to_datetime(cleverest_time, unit="s").strftime("%H:%M:%S")
            ),
            ("ClevFuzz", "Bug", ""): "",
            ("ClevFuzz", "Time", ""): "",
            ("ClevFuzz", "BugAll", ""): f"{clevfuzz_bugall}/{len_sub_data}",
        }
    )
agg_rows = pd.DataFrame(agg_rows)
# to multiindex
agg_rows.set_index(["Scenario", "Subject", "Issue"], inplace=True)
# display(agg_rows)
agg_rows.columns = pd.MultiIndex.from_tuples(agg_rows.columns)
display(agg_rows)

Scenario: BIC
Scenario: FIX


WAFLGo                   \
                           Init    Bug      Time   
                                                   
Scenario Subject   Issue                           
BIC      Aggregate               10/26  19:39:50   
FIX      Aggregate                8/24  21:48:32   

                                             Cleverest                       \
                                                 Score  Reach Change    Bug   
                                                                              
Scenario Subject   Issue                                                      
BIC      Aggregate        \SLIDER{3.3em}{0.51}{circle}  22/26  17/26   7/26   
FIX      Aggregate        \SLIDER{3.3em}{0.51}{circle}  19/24  14/24  11/24   

                                   ClevFuzz              
                              Time      Bug Time BugAll  
                                                         
Scenario Subject   Issue                                 
BIC      Aggregate        00:01:08                20/26  
FIX      Aggregate        00:00:51                18/24

In [25]:
# format
tb4_latex = tb4.copy()
## index formats
tb4_latex.index = tb4_latex.index.set_levels(
    [
        tb4_latex.index.levels[0],
        tb4_latex.index.levels[1].map(
            {
                **ALL_SUBJECTS_LATEX,
                "Aggregate": "Aggregate",
            }
        ),
        tb4_latex.index.levels[2].map(lambda x: f"\\#{x}" if x else ""),
    ],
)
tb4_latex.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4_latex.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(1),
            categories=[
                *ALL_SUBJECTS_LATEX.values(),
                "Aggregate",
            ],
            ordered=True,
        ),
        tb4_latex.index.get_level_values(2),
    ],
    names=["Scenario", "Subject", "Issue"],
)
## column formats
### ["WAFLGo", "Init"]:
###  "F" -> r"\xmark",
###  "B" -> r"\cmark",
###  "R" -> r"\rmark",
###  NaN -> "-"
tb4_latex[("WAFLGo", "Init")] = tb4_latex[("WAFLGo", "Init")].map(
    {"F": r"\xmark", "B": r"\cmark", "D": r"\dmark", "R": r"\rmark", np.nan: "-"}
)
### ["WAFLGo", "Bug"]:
###  if number, "{int(x)}/10"
###  if NaN, "-"
tb4_latex[("WAFLGo", "Bug")] = tb4_latex[("WAFLGo", "Bug")].map(
    lambda x: f"{int(x)}/10" if not pd.isna(x) else "-"
)
### ["WAFLGo", "Time"]:
###  if NaN, "-"
###  if 86400, "T.O."
###  else, hh:mm:ss
tb4_latex[("WAFLGo", "Time")] = tb4_latex[("WAFLGo", "Time")].map(
    lambda x: (
        "-"
        if pd.isna(x)
        else (
            "T.O."
            if x == 86400
            else str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
        )
    )
)
### ["Cleverest", "Score"]:
###  f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
tb4_latex[("Cleverest", "Score")] = tb4_latex[("Cleverest", "Score")].map(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
### ["Cleverest", "Bug"]:
###  "{int(x)}/10"
tb4_latex[("Cleverest", "Reach")] = tb4_latex[("Cleverest", "Reach")].map(
    lambda x: f"{int(x)}/10"
)
tb4_latex[("Cleverest", "Change")] = tb4_latex[("Cleverest", "Change")].map(
    lambda x: f"{int(x)}/10"
)
tb4_latex[("Cleverest", "Bug")] = tb4_latex[("Cleverest", "Bug")].map(
    lambda x: f"{int(x)}/10"
)
### ["Cleverest", "Time"]:
###  hh:mm:ss
tb4_latex[("Cleverest", "Time")] = tb4_latex[("Cleverest", "Time")].map(
    lambda x: str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
)
### ["ClevFuzz", "Bug"]:
###  if NaN, "-/-"
###  else, as it is
tb4_latex[("ClevFuzz", "Bug")] = tb4_latex[("ClevFuzz", "Bug")].map(
    lambda x: x if not pd.isna(x) else "-/-"
)
### ["ClevFuzz", "Time"]:
###  if NaN, "-"
###  if 86400, "T.O."
###  else, hh:mm:ss
tb4_latex[("ClevFuzz", "Time")] = tb4_latex[("ClevFuzz", "Time")].map(
    lambda x: (
        "-"
        if pd.isna(x)
        else (
            "T.O."
            if x == 86400
            else str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
        )
    )
)
### ["ClevFuzz", "BugAll"]:
### "{int(x)}/10"
tb4_latex[("ClevFuzz", "BugAll")] = tb4_latex[("ClevFuzz", "BugAll")].map(
    lambda x: f"{int(x)}/10"
)

# Add the aggregation row
tb4_latex.loc[("BIC", r"\multicolumn{2}{c|}{Aggregate}", ""), :] = tuple(
    agg_rows.loc[("BIC", "Aggregate", "")]
)
tb4_latex.loc[("FIX", r"\multicolumn{2}{c|}{Aggregate}", ""), :] = tuple(
    agg_rows.loc[("FIX", "Aggregate", "")]
)

# sort the index
tb4_latex.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4_latex.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(1),
            categories=[
                *ALL_SUBJECTS_LATEX.values(),
                r"\multicolumn{2}{c|}{Aggregate}",
            ],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(2),
            categories=tb4_latex.index.get_level_values(2).unique(),
            ordered=True,
        ),
    ],
    names=["Scenario", "Subject", "Issue"],
)
tb4_latex = tb4_latex.sort_index()

display(tb4_latex)
tb4_latex = (
    tb4_latex.to_latex()
    .replace("[t]", "")
    .replace(
        r"\multirow{24}{*}{BIC}",
        r"\multirow{24}{*}{\begin{sideways}Bug-finding\end{sideways}}",
    )
    .replace(
        r"\multirow{24}{*}{FIX}",
        r"\multirow{24}{*}{\begin{sideways}Bug-reproduction\end{sideways}}",
    )
    .replace(
        r"\multirow{15}{*}{BIC}",
        r"\multirow{15}{*}{\begin{sideways}Bug-finding\end{sideways}}",
    )
    .replace(
        r"\multirow{15}{*}{FIX}",
        r"\multirow{15}{*}{\begin{sideways}Bug-reproduction\end{sideways}}",
    )
    .replace(
        r"\multicolumn{2}{c|}{Average} &  &", r"\multicolumn{2}{c|}{Average} &"
    )
    # .replace(r"\cline{1-12} \cline{2-12}", r"\midrule")
    # .replace(r"\cline{2-12}", r"\cmidrule{2-12}")
    .replace(r"\cline{1-14} \cline{2-14}", r"\midrule")
    .replace(r"\cline{2-14}", r"\cmidrule{2-14}")
    .replace(
        r"\multicolumn{2}{c|}{Aggregate} &  &",
        r"\multicolumn{2}{c|}{Aggregate} &",
    )
)
print(tb4_latex)

WAFLGo                   \
                                                   Init    Bug      Time   
Scenario Subject                        Issue                              
BIC      \mujs                          \#65     \xmark  10/10  04:43:56   
                                        \#141    \xmark   0/10      T.O.   
                                        \#145    \xmark  10/10  00:17:21   
                                        \#166    \cmark  10/10      T.O.   
         \libxml                        \#535    \cmark  10/10      T.O.   
                                        \#550    \dmark   0/10      T.O.   
                                        \#553    \cmark  10/10      T.O.   
                                        \#720    \dmark   0/10      T.O.   
                                        \#841    \cmark  10/10      T.O.   
         \poppler                       \#1282   \rmark   6/10  00:06:23   
                                        \#1289   \rmark   0/10      T.O.   
                                        \#1303   \xmark   0/10      T.O.   
                                        \#1305   \xmark   0/10      T.O.   
                                        \#1381   \rmark   0/10      T.O.   
         \jerryscript                   \#5117   \rmark   4/10  00:13:02   
                                        \#5138   \rmark   0/10      T.O.   
                                        \#5153   \rmark   0/10      T.O.   
         \jq                            \#2825   \rmark   0/10      T.O.   
                                        \#2976   \rmark   2/10  01:55:03   
                                        \#3262   \cmark  10/10      T.O.   
                                        \#49014  \xmark   0/10      T.O.   
         \micropython                   \#13007  \rmark   0/10      T.O.   
                                        \#13041  \xmark   0/10      T.O.   
                                        \#17733  \rmark   0/10      T.O.   
                                        \#17815  \rmark   0/10      T.O.   
                                        \#17847  \xmark   0/10      T.O.   
         \multicolumn{2}{c|}{Aggregate}                  10/26  19:39:50   
FIX      \mujs                          \#65     \rmark   0/10      T.O.   
                                        \#141    \xmark   5/10  00:12:47   
                                        \#145    \xmark  10/10  00:08:26   
                                        \#166    \cmark  10/10      T.O.   
         \libxml                        \#535    \cmark  10/10      T.O.   
                                        \#550    \xmark   0/10      T.O.   
                                        \#553    \cmark  10/10      T.O.   
                                        \#720    \dmark   0/10      T.O.   
                                        \#841    \cmark  10/10      T.O.   
         \poppler                       \#1282   \rmark   0/10      T.O.   
                                        \#1289   \rmark   0/10      T.O.   
                                        \#1303   \xmark   0/10      T.O.   
                                        \#1305   \rmark   0/10      T.O.   
                                        \#1381   \rmark   0/10      T.O.   
         \jerryscript                   \#5117   \rmark   0/10      T.O.   
                                        \#5138   \xmark   0/10      T.O.   
                                        \#5013   \xmark   0/10      T.O.   
         \jq                            \#2825   \rmark   0/10      T.O.   
                                        \#2976   \rmark   1/10  19:03:38   
                                        \#3262   \cmark  10/10      T.O.   
                                        \#49014  \xmark   0/10      T.O.   
         \micropython                   \#13041  \dmark   0/10      T.O.   
                                        \#17733  \dmark   0/10      T.O.   
                     

\begin{tabular}{llllllllllllll}
\toprule
 &  &  & \multicolumn{3}{r}{WAFLGo} & \multicolumn{5}{r}{Cleverest} & \multicolumn{3}{r}{ClevFuzz} \\
 &  &  & Init & Bug & Time & Score & Reach & Change & Bug & Time & Bug & Time & BugAll \\
Scenario & Subject & Issue &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow{27}{*}{BIC} & \multirow{4}{*}{\mujs} & \#65 & \xmark & 10/10 & 04:43:56 & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 00:00:07 & -/- & - & 10/10 \\
 &  & \#141 & \xmark & 0/10 & T.O. & \SLIDER{3.3em}{0.67}{circle} & 10/10 & 10/10 & 0/10 & 00:00:45 & 10/10 & 03:25:05 & 10/10 \\
 &  & \#145 & \xmark & 10/10 & 00:17:21 & \SLIDER{3.3em}{0.87}{circle} & 10/10 & 8/10 & 8/10 & 00:00:37 & 2/2 & 00:00:00 & 10/10 \\
 &  & \#166 & \cmark & 10/10 & T.O. & \SLIDER{3.3em}{0.37}{circle} & 10/10 & 1/10 & 0/10 & 00:00:57 & 0/10 & T.O. & 0/10 \\
\cmidrule{2-14}
 & \multirow{5}{*}{\libxml} & \#535 & \cmark & 10/10 & T.O. & \SLIDER{3.3em}{1.00}{circle} & 10/10 & 10/10 & 10/10 & 00:00:0

# Figure 3

In [26]:
clevfuzz_data

commit          idx  success       time cleverest  \
scenario subject     iid                                                        
BIC      jerryscript 5013  b7e3bae  250826-2133        0  86400.000         R   
                     5013  b7e3bae  250826-2139        0  86400.000         R   
                     5013  b7e3bae  250826-2145        0  86400.000         X   
                     5013  b7e3bae  250826-2150        1    101.758         R   
                     5013  b7e3bae  250826-2156        0  86400.000         R   
...                            ...          ...      ...        ...       ...   
FIX      z3          7252  a6b5027  250826-2135        0  86400.000         R   
                     7252  a6b5027  250826-2140        0  86400.000         R   
                     7252  a6b5027  250826-2150        0  86400.000         R   
                     7252  a6b5027  250903-2341        0  86400.000         R   
                     7252  a6b5027  250903-2351        0  86400.000         R   

                           try  
scenario subject     iid        
BIC      jerryscript 5013    1  
                     5013    1  
                     5013    1  
                     5013    1  
                     5013    1  
...                        ...  
FIX      z3          7252    1  
                     7252    1  
                     7252    1  
                     7252    1  
                     7252    1  

[328 rows x 6 columns]

In [27]:
clevfuzz_data[["cleverest", "success"]]

data = {}
for scenario in ["BIC", "FIX"]:
    sub_data = clevfuzz_data.loc[scenario]
    # count the number of R in cleverest and 1 in success
    R_B = len(
        sub_data[
            (sub_data["cleverest"] == "R") & (sub_data["success"] == 1)
        ]
    )
    D_B = len(
        sub_data[
            (sub_data["cleverest"] == "D") & (sub_data["success"] == 1)
        ]
    )
    R_F = len(
        sub_data[
            (sub_data["cleverest"] == "R") & (sub_data["success"] == 0)
        ]
    )
    D_F = len(
        sub_data[
            (sub_data["cleverest"] == "D") & (sub_data["success"] == 0)
        ]
    )
    data[scenario] = {
        "R_B": R_B,
        "D_B": D_B,
        "R_F": R_F,
        "D_F": D_F,
    }
clevfuzz_convert = pd.DataFrame(data).T
clevfuzz_convert["B"] = clevfuzz_convert["R_B"] + clevfuzz_convert["D_B"]
clevfuzz_convert["F"] = clevfuzz_convert["R_F"] + clevfuzz_convert["D_F"]
# column order = ["R_B", "D_B", "B", "R_F", "D_F", "F"]
clevfuzz_convert = clevfuzz_convert[
    ["R_B", "D_B", "B", "R_F", "D_F", "F"]
]
clevfuzz_convert.loc["sum"] = clevfuzz_convert.sum()
clevfuzz_convert

,R_B,D_B,B,R_F,D_F,F
BIC,46,45,91,78,37,115
FIX,35,19,54,53,2,55
sum,81,64,145,131,39,170


In [28]:
tb4_time = tb4.copy()
# NOTE: depending on whether tb4 includes WAFLGo unrunnable instances, avg time corresponds to main/supp table
## total time = cleverest time if clevfuzz time is NaN, else sum of cleverest and clevfuzz time
tb4_time["ClevFuzz", "TotalTime"] = tb4_time["Cleverest"]["Time"]
tb4_time.loc[
    tb4_time["ClevFuzz"]["Time"].notna(), ("ClevFuzz", "TotalTime")
] = (tb4_time["Cleverest"]["Time"] + tb4_time["ClevFuzz"]["Time"])
display(tb4_time)
# average time for each scenario
tb4_time_avg = tb4_time.groupby("Scenario").agg(
    {("WAFLGo", "Time"): "mean", ("ClevFuzz", "TotalTime"): "mean"}
)
# format to hh:mm:ss
tb4_time_avg = tb4_time_avg.map(
    lambda x: str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
)
tb4_time_avg

WAFLGo                   Cleverest               \
                             Init   Bug        Time     Score Reach Change   
Scenario Subject     Issue                                                   
BIC      mujs        65         F  10.0  17036.2719       3.0    10     10   
                     141        F   0.0  86400.0000       2.0    10     10   
                     145        F  10.0   1041.3056       2.6    10      8   
                     166        B  10.0  86400.0000       1.1    10      1   
         libxml2     535        B  10.0  86400.0000       3.0    10     10   
                     550        D   0.0  86400.0000       1.2    10      2   
                     553        B  10.0  86400.0000       3.0    10     10   
                     720        D   0.0  86400.0000       1.0    10      0   
                     841        B  10.0  86400.0000       3.0    10     10   
         poppler     1282       R   6.0    383.0208       1.1     8      3   
                     1289       R   0.0  86400.0000       0.4     4      0   
                     1303       F   0.0  86400.0000       0.0     0      0   
                     1305       F   0.0  86400.0000       0.0     0      0   
                     1381       R   0.0  86400.0000       0.0     0      0   
         jerryscript 5117       R   4.0    782.7990       2.0    10     10   
                     5138       R   0.0  86400.0000       2.2    10     10   
                     5153       R   0.0  86400.0000       2.0    10     10   
         jq          2825       R   0.0  86400.0000       1.0    10      0   
                     2976       R   2.0   6903.9730       1.0    10      0   
                     3262       B  10.0  86400.0000       3.0    10     10   
                     49014      F   0.0  86400.0000       0.4     2      2   
         micropython 13007      R   0.0  86400.0000       1.6     9      7   
                     13041      F   0.0  86400.0000       0.0     0      0   
                     17733      R   0.0  86400.0000       1.0    10      0   
                     17815      R   0.0  86400.0000       1.9    10      9   
                     17847      F   0.0  86400.0000       2.0    10     10   
FIX      mujs        65         R   0.0  86400.0000       2.2    10      6   
                     141        F   5.0    767.1130       2.1    10     10   
                     145        F  10.0    506.1808       2.8    10      9   
                     166        B  10.0  86400.0000       3.0    10     10   
         libxml2     535        B  10.0  86400.0000       3.0    10     10   
                     550        F   0.0  86400.0000       0.0     0      0   
                     553        B  10.0  86400.0000       3.0    10     10   
                     720        D   0.0  86400.0000       0.7     7      0   
                     841        B  10.0  86400.0000       3.0    10     10   
         poppler     1282       R   0.0  86400.0000       0.9     9      0   
                     1289       R   0.0  86400.0000       2.0     8      6   
                     1303       F   0.0  86400.0000       0.2     1      1   
                     1305       R   0.0  86400.0000       0.0     0      0   
                     1381       R   0.0  86400.0000       0.0     0      0   
         jerryscript 5013       F   0.0  86400.0000       1.0    10      0   
                     5117       R   0.0  86400.0000       2.0    10      5   
                     5138       F   0.0  86400.0000       1.0    10      0   
         jq          2825       R   0.0  86400.0000       1.1    10      1   
                     2976       R   1.0  68618.2100       3.0    10     10   
                     3262       B  10.0  86400.0000       3.0    10     10   
                     49014      F   0.0  86400.0000       0.0     0      0   
         micropython 13041      D   0.0  86400.0000       2.0    10     10   
                     17733      D   0.0  86400.00

,WAFLGo,ClevFuzz
,Time,TotalTime
Scenario,,
BIC,19:39:50,05:42:01
FIX,21:48:32,06:11:24


# Comment statistics

In [29]:
# read columns msgwords,msgchars,difflines,diffchars from csv
stats_path = "stat_72commits.csv"
df_stats = pd.read_csv(stats_path)
df_notz3fix = df_stats[~((df_stats["proj"] == "z3") & (df_stats["scenario"] == "FIX"))]
display(df_stats.describe())
print(df_notz3fix["msgwords"].mean())

df_bic_stats = df_stats[df_stats["scenario"] == "BIC"]
df_fix_stats = df_stats[df_stats["scenario"] == "FIX"]
display(df_bic_stats.describe())
display(df_fix_stats.describe())

,iid,msgwords,msgchars,difflines,diffchars
count,72.000000,72.000000,72.000000,72.000000,72.000000
mean,7855.250000,36.111111,252.472222,272.152778,10356.416667
std,9652.534628,42.573071,275.107404,634.412841,25305.349553
min,65.000000,2.000000,10.000000,12.000000,411.000000
25%,1171.750000,9.750000,74.000000,21.500000,724.000000
50%,5065.000000,20.000000,163.500000,65.500000,2743.000000
75%,13975.000000,44.500000,309.750000,234.000000,6915.750000
max,49014.000000,208.000000,1428.000000,3291.000000,139757.000000


38.0


,iid,msgwords,msgchars,difflines,diffchars
count,36.00000,36.000000,36.000000,36.000000,36.000000
mean,7855.25000,43.638889,303.750000,493.722222,18852.916667
std,9721.23681,50.406435,323.486487,843.276352,33844.214984
min,65.00000,4.000000,41.000000,26.000000,882.000000
25%,1171.75000,12.750000,115.500000,85.750000,3135.000000
50%,5065.00000,21.500000,186.500000,205.500000,5929.000000
75%,13975.00000,54.750000,338.750000,479.250000,18439.000000
max,49014.00000,208.000000,1428.000000,3291.000000,139757.000000


,iid,msgwords,msgchars,difflines,diffchars
count,36.00000,36.000000,36.000000,36.000000,36.000000
mean,7855.25000,28.583333,201.194444,50.583333,1859.916667
std,9721.23681,31.927037,208.513764,65.972451,2253.967161
min,65.00000,2.000000,10.000000,12.000000,411.000000
25%,1171.75000,8.750000,67.000000,13.000000,518.750000
50%,5065.00000,17.500000,125.500000,21.000000,694.000000
75%,13975.00000,40.500000,297.000000,46.500000,2329.250000
max,49014.00000,152.000000,979.000000,280.000000,10230.000000
